# 03. NGCF-подобная модель с весами рейтингов

Ноутбук с экспериментами для графовой модели, где рейтинг влияет на силу связи между пользователем и фильмом.

Основной фокус:
- построение user-item графа;
- использование рейтинга как веса ребра;
- сравнение разных функций веса;
- проверка NGCF-подобного распространения сигнала;
- оценка в warm/cold сценариях.

Варианты функций веса в текущем ноутбуке:
- `linear_norm`;
- `exp_mixture`;
- `softplus_spline`.

Outputs очищены, чтобы ноутбук было удобнее хранить в репозитории.


In [ ]:
from __future__ import annotations

In [ ]:
try:
    import comet_ml  # noqa: F401
except Exception:
    try:
        get_ipython().run_line_magic("pip", "install -q comet_ml")
    except Exception as exc:
        print(f"[warning] comet_ml install skipped: {exc}")

In [ ]:
import json
import math
import os
import random
import time
from collections import defaultdict
from contextlib import nullcontext
from dataclasses import asdict, dataclass, field, is_dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    from comet_ml import Experiment
    COMET_AVAILABLE = True
except Exception:
    Experiment = None
    COMET_AVAILABLE = False

In [ ]:
@dataclass
class PathsConfig:
    dataset_dir: str = "/kaggle/input/datasets/daniilzagatin/movielens20-withfeatures-split"
    output_dir: str = "/kaggle/working/explicit_rating_baselines"


@dataclass
class FeatureConfig:
    use_item_features: bool = True
    item_feature_mode: str = "init_only"   # none | add | project_only | concat_mlp | init_only
    scaler_kind: str = "standard"
    fit_scaler_on: str = "warm"      # warm | all
    feature_dropout: float = 0.10
    feature_hidden_dim: int = 256
    init_only_svd_n_iter: int = 7
    init_only_random_state: int = 42


@dataclass
class TrainConfig:
    seed: int = 42
    device: str = "cuda:0" if torch.cuda.is_available() else "cpu"
    epochs: int = 10
    lr: float = 3e-4
    weight_decay: float = 1e-5
    batch_size: int = 1024
    num_workers: int = 4 if (os.cpu_count() or 0) >= 4 else 2
    prefetch_factor: int = 2
    persistent_workers: bool = True
    patience: int = 3
    grad_clip_norm: Optional[float] = 5.0
    log_every_n_steps: int = 100
    verbose: bool = True
    positive_threshold: float = 4.0
    train_rating_weighting: str = "linear_norm"   # none | binary | raw | linear_norm | centered
    max_history: int = 50
    min_history_for_sequence: int = 1
    neg_samples: int = 1
    relation_edge_batch_size: int = 131072
    graph_loss_batches_per_epoch_log: int = 10
    use_amp: bool = True
    tf32: bool = True
    use_multi_gpu: bool = True
    compile_model: bool = False
    compile_mode: str = "default"
    pin_memory: bool = True
    relation_backward_per_batch: bool = True
    empty_cache_each_epoch: bool = False


@dataclass
class EvalConfig:
    split_name: str = "val"  # kept for backward compatibility
    ks: Tuple[int, ...] = (10, 20, 50)
    metric_gain_types: Tuple[str, ...] = ("binary_ge_4", "centered_3", "power")
    binary_metric_names: Tuple[str, ...] = ("precision", "recall", "hitrate", "mrr", "map", "ndcg")
    graded_metric_names: Tuple[str, ...] = ("ndcg",)
    positive_threshold: float = 4.0
    neutral_rating: float = 3.0
    power_beta: float = 0.2
    power_gamma: float = 2.0
    exclude_seen_items: bool = True
    user_batch_size: int = 256
    item_batch_size: Optional[int] = None
    skip_users_with_no_relevant_targets: bool = True
    compute_loss: bool = True
    loss_batch_size: int = 4096
    loss_max_rows: Optional[int] = None
    score_mode: str = "default"  # default | expected_rating | prob_positive
    main_metric_name: str = "ndcg"
    main_metric_gain_type: str = "centered_3"
    main_metric_k: int = 10
    eval_scenarios: Tuple[str, ...] = ("warm", "cold_user", "cold_item", "both_cold")
    evaluate_cold_on_val: bool = True
    evaluate_cold_on_test: bool = True
    cold_user_use_support_all: bool = True


@dataclass
class CometConfig:
    enabled: bool = True
    project_name: str = "explicit-movie-ranking"
    workspace: Optional[str] = None
    experiment_name: Optional[str] = None
    tags: List[str] = field(default_factory=lambda: ["warm", "baseline"])
    log_code: bool = False


@dataclass
class GraphConfig:
    num_layers: int = 2
    embedding_dim: int = 128
    hidden_dim: int = 128
    dropout: float = 0.10
    relation_aggregation: str = "sum"   # sum | mean
    relation_apply_time_decay: bool = False
    time_weight_fn: str = "exp"
    time_weight_gamma: float = 1e-8

    # Для method 1 / support pooling:
    # - relation_gcmc: обычно "none", так как рейтинг идет как relation type
    # - support_gcmc / two-tower history: может использоваться как sample/history weight
    #
    # Для ordinal_ngcf это главный переключатель функции веса ребра:
    #   linear_norm | exp_mixture | softplus_spline
    rating_weight_fn: str = "linear_norm"

    relation_score_mode: str = "expected_rating"  # expected_rating | prob_positive
    aux_bpr_weight: float = 0.0
    support_history_pooling: str = "weighted_mean"   # mean | weighted_mean | attention
    support_use_rating: bool = True
    support_use_time: bool = True
    support_time_weight_fn: str = "exp"
    support_time_weight_gamma: float = 1e-8
    support_alignment_weight: float = 0.2
    support_alignment_loss: str = "mse"   # mse | cosine
    support_user_batch_size: int = 4096
    checkpoint_encoder_layers: bool = True

    # =========================
    # Ordinal NGCF / method 2
    # =========================
    rating_r_min: float = 0.5
    rating_r_max: float = 5.0
    rating_weight_eps: float = 1e-3

    # exp_mixture: smooth learnable monotone curve
    rating_exp_terms: int = 4
    rating_exp_lambda_clip: float = 8.0

    # softplus_spline: flexible learnable monotone curve
    rating_softplus_num_knots: int = 5
    rating_softplus_init_kappa: float = 8.0
    rating_softplus_learn_kappa: bool = True

    # NGCF encoder
    ngcf_layer_pooling: str = "concat"  # concat | mean
    ngcf_use_l2_norm: bool = True

    # Fixed cold-user initialization, no additional support encoder.
    # positive_item_mean: one shared vector = weighted mean of train positive item embeddings.
    # mean_warm_user: one shared vector = mean of warm user ID embeddings.
    # zero: one shared zero vector.
    cold_user_init_strategy: str = "positive_item_mean"  # positive_item_mean | mean_warm_user | zero


@dataclass
class TwoTowerConfig:
    embedding_dim: int = 128
    hidden_dim: int = 256
    dropout: float = 0.10
    use_user_id_embedding: bool = True
    use_history: bool = True
    history_pooling: str = "weighted_mean"   # mean | weighted_mean | attention
    use_rating_in_history: bool = True
    use_time_in_history: bool = True
    time_weight_fn: str = "exp"
    time_weight_gamma: float = 1e-8
    rating_weight_fn: str = "linear_norm"


@dataclass
class MFConfig:
    embedding_dim: int = 128
    dropout: float = 0.0


@dataclass
class ModelConfig:
    model_type: str = "mf"   # mf | two_tower | relation_gcmc | support_gcmc | ordinal_ngcf
    mf: MFConfig = field(default_factory=MFConfig)
    two_tower: TwoTowerConfig = field(default_factory=TwoTowerConfig)
    graph: GraphConfig = field(default_factory=GraphConfig)


@dataclass
class ExperimentConfig:
    paths: PathsConfig = field(default_factory=PathsConfig)
    features: FeatureConfig = field(default_factory=FeatureConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    eval: EvalConfig = field(default_factory=EvalConfig)
    comet: CometConfig = field(default_factory=CometConfig)
    model: ModelConfig = field(default_factory=ModelConfig)


cfg = ExperimentConfig()
# This notebook is focused on method 2: Ordinal NGCF.
cfg.model.model_type = "ordinal_ngcf"
# Other implemented models are kept for compatibility:
# cfg.model.model_type = "mf"
# cfg.model.model_type = "two_tower"
# cfg.model.model_type = "relation_gcmc"
# cfg.model.model_type = "support_gcmc"

# Recommended first sweeps:
# cfg.train.batch_size = 512
# cfg.model.two_tower.embedding_dim = 192
# cfg.train.epochs = 10
# cfg.train.patience = 4
cfg

In [ ]:
def dc_to_dict(obj):
    if is_dataclass(obj):
        return {k: dc_to_dict(v) for k, v in asdict(obj).items()}
    if isinstance(obj, dict):
        return {k: dc_to_dict(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [dc_to_dict(v) for v in obj]
    return obj


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def configure_runtime(cfg: ExperimentConfig) -> None:
    if torch.cuda.is_available():
        if cfg.train.tf32:
            try:
                torch.set_float32_matmul_precision("high")
            except Exception:
                pass
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True


def use_amp(cfg: ExperimentConfig, device: torch.device) -> bool:
    if not (cfg.train.use_amp and device.type == "cuda"):
        return False
    if cfg.model.model_type in {"relation_gcmc", "support_gcmc", "ordinal_ngcf"}:
        return False
    return True


def get_autocast_context(cfg: ExperimentConfig, device: torch.device):
    if use_amp(cfg, device):
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def make_grad_scaler(cfg: ExperimentConfig, device: torch.device):
    if device.type == "cuda":
        return torch.amp.GradScaler("cuda", enabled=use_amp(cfg, device))
    return torch.amp.GradScaler("cpu", enabled=False)


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, nn.DataParallel) else model


def compute_grad_norm(parameters: Iterable[torch.nn.Parameter]) -> float:
    total_sq = 0.0
    for p in parameters:
        if p.grad is None:
            continue
        grad = p.grad.detach()
        total_sq += float(torch.sum(grad * grad).item())
    return float(total_sq ** 0.5)


def get_gpu_info() -> Dict[str, object]:
    if not torch.cuda.is_available():
        return {"num_gpus": 0, "gpu_names": []}
    return {
        "num_gpus": torch.cuda.device_count(),
        "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    }


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(data: dict, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json(path: str | Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def rating_weight(rating: np.ndarray | torch.Tensor, kind: str, r_min: float = 0.5, r_max: float = 5.0, neutral: float = 3.0):
    if isinstance(rating, torch.Tensor):
        x = rating.float()
        if kind == "none":
            return torch.ones_like(x)
        if kind == "binary":
            return (x >= 4.0).float()
        if kind == "raw":
            return x
        if kind == "linear_norm":
            return torch.clamp((x - r_min) / max(r_max - r_min, 1e-8), min=0.0)
        if kind == "centered":
            return torch.clamp(x - neutral, min=0.0)
        raise ValueError(f"Unknown rating weight kind: {kind}")
    x = np.asarray(rating, dtype=np.float32)
    if kind == "none":
        return np.ones_like(x)
    if kind == "binary":
        return (x >= 4.0).astype(np.float32)
    if kind == "raw":
        return x
    if kind == "linear_norm":
        return np.clip((x - r_min) / max(r_max - r_min, 1e-8), a_min=0.0, a_max=None).astype(np.float32)
    if kind == "centered":
        return np.clip(x - neutral, a_min=0.0, a_max=None).astype(np.float32)
    raise ValueError(f"Unknown rating weight kind: {kind}")


def time_weight(delta_seconds: np.ndarray | torch.Tensor, kind: str, gamma: float = 1e-8):
    if isinstance(delta_seconds, torch.Tensor):
        x = delta_seconds.float()
        if kind == "none":
            return torch.ones_like(x)
        if kind == "exp":
            return torch.exp(-gamma * x)
        if kind == "inverse":
            return 1.0 / (1.0 + gamma * x)
        if kind == "sqrt_inverse":
            return 1.0 / torch.sqrt(1.0 + gamma * x)
        raise ValueError(f"Unknown time weight kind: {kind}")
    x = np.asarray(delta_seconds, dtype=np.float32)
    if kind == "none":
        return np.ones_like(x)
    if kind == "exp":
        return np.exp(-gamma * x).astype(np.float32)
    if kind == "inverse":
        return (1.0 / (1.0 + gamma * x)).astype(np.float32)
    if kind == "sqrt_inverse":
        return (1.0 / np.sqrt(1.0 + gamma * x)).astype(np.float32)
    raise ValueError(f"Unknown time weight kind: {kind}")


def gain_from_rating(
    ratings: np.ndarray,
    gain_type: str,
    positive_threshold: float = 4.0,
    neutral_rating: float = 3.0,
    power_beta: float = 0.2,
    power_gamma: float = 2.0,
) -> np.ndarray:
    ratings = np.asarray(ratings, dtype=np.float32)
    if gain_type == "binary_ge_4":
        return (ratings >= positive_threshold).astype(np.float32)
    if gain_type == "centered_3":
        return np.clip(ratings - neutral_rating, a_min=0.0, a_max=None).astype(np.float32)
    if gain_type == "power":
        x = np.clip((ratings - 1.0) / 4.0, a_min=0.0, a_max=1.0)
        return (power_beta + (1.0 - power_beta) * np.power(x, power_gamma)).astype(np.float32)
    raise ValueError(f"Unknown gain_type={gain_type}")


def get_metric_gain_types(cfg_eval: EvalConfig) -> Tuple[str, ...]:
    gain_types = tuple(dict.fromkeys(cfg_eval.metric_gain_types or (cfg_eval.main_metric_gain_type,)))
    if not gain_types:
        gain_types = (cfg_eval.main_metric_gain_type,)
    return gain_types


def gain_type_metric_family(gain_type: str) -> str:
    if gain_type == "binary_ge_4":
        return "binary"
    if gain_type in {"centered_3", "power"}:
        return "graded"
    raise ValueError(f"Unknown gain_type={gain_type}")

In [ ]:
@dataclass
class PreparedData:
    train_df: pd.DataFrame
    val_df: pd.DataFrame
    test_df: pd.DataFrame
    eval_frames: Dict[str, pd.DataFrame]
    item_features: torch.Tensor
    item_feature_cols: List[str]
    num_users: int
    num_warm_users: int
    num_items: int
    num_warm_items: int
    warm_user2idx: Dict[str, int]
    warm_item2idx: Dict[str, int]
    all_item2idx: Dict[str, int]
    cold_user2idx: Dict[str, int]
    cold_item2idx: Dict[str, int]
    trainable_item_mask: torch.Tensor
    user_histories: Dict[int, List[Tuple[int, float, int]]]
    eval_histories: Dict[str, Dict[int, List[Tuple[int, float, int]]]]
    seen_items: Dict[int, set]
    eval_seen_items: Dict[str, Dict[int, set]]
    unique_ratings: List[float]
    rating2class: Dict[float, int]
    class2rating: Dict[int, float]
    output_dir: Path
    split_config: dict


REQUIRED_FILES = [
    "train_warm_interactions.parquet",
    "warm_val_interactions.parquet",
    "warm_test_interactions.parquet",
    "warm_user2idx.json",
    "warm_item2idx.json",
    "feature_cols.json",
]


def auto_find_dataset_dir(required_files: Sequence[str], preferred_path: str) -> Path:
    preferred = Path(preferred_path)
    if preferred.exists() and all((preferred / fname).exists() for fname in required_files):
        return preferred
    search_roots = [Path("/kaggle/input"), Path("/mnt/data"), Path(".")]
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob(required_files[0]):
            candidate = path.parent
            if all((candidate / fname).exists() for fname in required_files):
                return candidate
    raise FileNotFoundError(
        f"Could not find dataset directory containing: {required_files}. Preferred path: {preferred_path}"
    )


def load_interactions(path: Path, user2idx: dict, item2idx: dict) -> pd.DataFrame:
    df = pd.read_parquet(path)
    if "user_idx" not in df.columns:
        df["user_idx"] = df["userId"].astype(str).map(user2idx)
    if "item_idx" not in df.columns:
        df["item_idx"] = df["movieId"].astype(str).map(item2idx)
    df = df.dropna(subset=["user_idx", "item_idx"]).copy()
    df["user_idx"] = df["user_idx"].astype(int)
    df["item_idx"] = df["item_idx"].astype(int)
    df["rating"] = df["rating"].astype(float)
    if "timestamp" in df.columns:
        df["timestamp"] = df["timestamp"].astype(np.int64)
    else:
        df["timestamp"] = 0
    return df


def prepare_item_feature_matrix(
    dataset_dir: Path,
    feature_cfg: FeatureConfig,
    all_item2idx: dict,
    warm_item_keys: Sequence[str],
    output_dir: Path,
) -> Tuple[torch.Tensor, List[str]]:
    feature_cols = load_json(dataset_dir / "feature_cols.json")
    feat_path = dataset_dir / "item_features_all.parquet"
    if not feat_path.exists():
        feat_path = dataset_dir / "item_features_warm.parquet"
    feat_df = pd.read_parquet(feat_path)
    feat_df["movieId"] = feat_df["movieId"].astype(str)
    feat_df = feat_df[feat_df["movieId"].isin(set(all_item2idx.keys()))].copy()
    feat_df["item_idx"] = feat_df["movieId"].map(all_item2idx).astype(int)
    feat_df = feat_df.sort_values("item_idx")

    for col in feature_cols:
        if col not in feat_df.columns:
            feat_df[col] = 0.0

    aligned = pd.DataFrame({"item_idx": np.arange(len(all_item2idx), dtype=np.int64)})
    aligned = aligned.merge(feat_df[["item_idx", *feature_cols]], on="item_idx", how="left")
    aligned[feature_cols] = aligned[feature_cols].fillna(0.0)

    x = aligned[feature_cols].astype(np.float32).to_numpy()
    if feature_cfg.scaler_kind == "standard":
        scaler = StandardScaler(with_mean=True, with_std=True)
        if feature_cfg.fit_scaler_on == "warm":
            warm_idx = sorted(int(all_item2idx[str(k)]) for k in warm_item_keys if str(k) in all_item2idx)
            fit_x = x[warm_idx] if warm_idx else x
        elif feature_cfg.fit_scaler_on == "all":
            fit_x = x
        else:
            raise ValueError(f"Unsupported fit_scaler_on={feature_cfg.fit_scaler_on}")
        scaler.fit(fit_x)
        x = scaler.transform(x).astype(np.float32)
        scaler_stats = {
            "mean": scaler.mean_.tolist(),
            "scale": scaler.scale_.tolist(),
            "feature_cols": feature_cols,
            "fit_scaler_on": feature_cfg.fit_scaler_on,
        }
        save_json(scaler_stats, output_dir / "feature_scaler_stats.json")
    elif feature_cfg.scaler_kind == "none":
        x = x.astype(np.float32)
    else:
        raise ValueError(f"Unsupported scaler_kind={feature_cfg.scaler_kind}")
    return torch.tensor(x, dtype=torch.float32), feature_cols


def build_user_histories(train_df: pd.DataFrame) -> Tuple[Dict[int, List[Tuple[int, float, int]]], Dict[int, set]]:
    histories: Dict[int, List[Tuple[int, float, int]]] = defaultdict(list)
    seen_items: Dict[int, set] = defaultdict(set)
    if train_df.empty:
        return {}, {}
    sort_df = train_df.sort_values(["user_idx", "timestamp", "item_idx"]).reset_index(drop=True)
    for row in sort_df.itertuples(index=False):
        histories[int(row.user_idx)].append((int(row.item_idx), float(row.rating), int(row.timestamp)))
        seen_items[int(row.user_idx)].add(int(row.item_idx))
    return dict(histories), dict(seen_items)


def load_optional_interactions(path: Path, user2idx: dict, item2idx: dict) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=["userId", "movieId", "rating", "timestamp", "user_idx", "item_idx"])
    return load_interactions(path, user2idx, item2idx)


def load_prepared_data(cfg: ExperimentConfig) -> PreparedData:
    dataset_dir = auto_find_dataset_dir(REQUIRED_FILES, cfg.paths.dataset_dir)
    output_dir = ensure_dir(cfg.paths.output_dir)
    print(f"Loading dataset from: {dataset_dir}")

    warm_user2idx = {str(k): int(v) for k, v in load_json(dataset_dir / "warm_user2idx.json").items()}
    warm_item2idx = {str(k): int(v) for k, v in load_json(dataset_dir / "warm_item2idx.json").items()}
    if (dataset_dir / "all_item2idx.json").exists():
        all_item2idx = {str(k): int(v) for k, v in load_json(dataset_dir / "all_item2idx.json").items()}
    else:
        all_item2idx = dict(warm_item2idx)
    cold_user2idx_local = {str(k): int(v) for k, v in load_json(dataset_dir / "cold_user2idx.json").items()} if (dataset_dir / "cold_user2idx.json").exists() else {}
    cold_item2idx = {str(k): int(v) for k, v in load_json(dataset_dir / "cold_item2idx.json").items()} if (dataset_dir / "cold_item2idx.json").exists() else {}
    split_config = load_json(dataset_dir / "split_config.json") if (dataset_dir / "split_config.json").exists() else {}

    train_df = load_interactions(dataset_dir / "train_warm_interactions.parquet", warm_user2idx, all_item2idx)
    val_df = load_interactions(dataset_dir / "warm_val_interactions.parquet", warm_user2idx, all_item2idx)
    test_df = load_interactions(dataset_dir / "warm_test_interactions.parquet", warm_user2idx, all_item2idx)

    item_features, feature_cols = prepare_item_feature_matrix(
        dataset_dir,
        cfg.features,
        all_item2idx,
        warm_item_keys=list(warm_item2idx.keys()),
        output_dir=output_dir,
    )

    user_histories, seen_items = build_user_histories(train_df)
    warm_item_indices = sorted(int(all_item2idx[str(k)]) for k in warm_item2idx.keys() if str(k) in all_item2idx)
    trainable_item_mask = torch.zeros(len(all_item2idx), dtype=torch.float32)
    if warm_item_indices:
        trainable_item_mask[torch.tensor(warm_item_indices, dtype=torch.long)] = 1.0

    unique_ratings = sorted(float(x) for x in train_df["rating"].unique().tolist())
    rating2class = {r: i for i, r in enumerate(unique_ratings)}
    class2rating = {i: r for r, i in rating2class.items()}

    eval_frames: Dict[str, pd.DataFrame] = {
        "warm_val": val_df,
        "warm_test": test_df,
    }
    eval_histories: Dict[str, Dict[int, List[Tuple[int, float, int]]]] = {
        "warm_val": user_histories,
        "warm_test": user_histories,
    }
    eval_seen_items: Dict[str, Dict[int, set]] = {
        "warm_val": seen_items,
        "warm_test": seen_items,
    }

    if cold_user2idx_local:
        cold_user2idx = {raw_uid: len(warm_user2idx) + int(local_idx) for raw_uid, local_idx in cold_user2idx_local.items()}
        support_name = "cold_user_support_all.parquet" if cfg.eval.cold_user_use_support_all else "cold_user_support.parquet"
        cold_support_df = load_optional_interactions(dataset_dir / support_name, cold_user2idx, all_item2idx)
        cold_histories, cold_seen_items = build_user_histories(cold_support_df)

        cold_user_val_df = load_optional_interactions(dataset_dir / "cold_user_val_interactions.parquet", cold_user2idx, all_item2idx)
        cold_user_test_df = load_optional_interactions(dataset_dir / "cold_user_test_interactions.parquet", cold_user2idx, all_item2idx)
        both_cold_val_df = load_optional_interactions(dataset_dir / "both_cold_val_interactions.parquet", cold_user2idx, all_item2idx)
        both_cold_test_df = load_optional_interactions(dataset_dir / "both_cold_test_interactions.parquet", cold_user2idx, all_item2idx)

        eval_frames.update({
            "cold_user_val": cold_user_val_df,
            "cold_user_test": cold_user_test_df,
            "both_cold_val": both_cold_val_df,
            "both_cold_test": both_cold_test_df,
        })
        eval_histories.update({
            "cold_user_val": cold_histories,
            "cold_user_test": cold_histories,
            "both_cold_val": cold_histories,
            "both_cold_test": cold_histories,
        })
        eval_seen_items.update({
            "cold_user_val": cold_seen_items,
            "cold_user_test": cold_seen_items,
            "both_cold_val": cold_seen_items,
            "both_cold_test": cold_seen_items,
        })
    else:
        cold_user2idx = {}

    cold_item_val_df = load_optional_interactions(dataset_dir / "cold_item_val_interactions.parquet", warm_user2idx, all_item2idx)
    cold_item_test_df = load_optional_interactions(dataset_dir / "cold_item_test_interactions.parquet", warm_user2idx, all_item2idx)
    eval_frames.update({
        "cold_item_val": cold_item_val_df,
        "cold_item_test": cold_item_test_df,
    })
    eval_histories.update({
        "cold_item_val": user_histories,
        "cold_item_test": user_histories,
    })
    eval_seen_items.update({
        "cold_item_val": seen_items,
        "cold_item_test": seen_items,
    })

    return PreparedData(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        eval_frames=eval_frames,
        item_features=item_features,
        item_feature_cols=feature_cols,
        num_users=len(warm_user2idx) + len(cold_user2idx),
        num_warm_users=len(warm_user2idx),
        num_items=len(all_item2idx),
        num_warm_items=len(warm_item_indices),
        warm_user2idx=warm_user2idx,
        warm_item2idx=warm_item2idx,
        all_item2idx=all_item2idx,
        cold_user2idx=cold_user2idx,
        cold_item2idx=cold_item2idx,
        trainable_item_mask=trainable_item_mask,
        user_histories=user_histories,
        eval_histories=eval_histories,
        seen_items=seen_items,
        eval_seen_items=eval_seen_items,
        unique_ratings=unique_ratings,
        rating2class=rating2class,
        class2rating=class2rating,
        output_dir=output_dir,
        split_config=split_config,
    )


set_seed(cfg.train.seed)
data = load_prepared_data(cfg)
print(data.train_df.head())
print(f"num_users={data.num_users:,} | num_items={data.num_items:,}")
print(f"train={len(data.train_df):,} | val={len(data.val_df):,} | test={len(data.test_df):,}")
print(f"feature_dim={data.item_features.shape[1]}")
print(f"rating values={data.unique_ratings}")

In [ ]:
class PairwiseTrainDataset(Dataset):
    def __init__(self, train_df: pd.DataFrame, seen_items: Dict[int, set], candidate_items: Sequence[int], positive_threshold: float):
        pos_df = train_df[train_df["rating"] >= positive_threshold].copy()
        self.users = pos_df["user_idx"].astype(np.int64).to_numpy()
        self.items = pos_df["item_idx"].astype(np.int64).to_numpy()
        self.ratings = pos_df["rating"].astype(np.float32).to_numpy()
        self.seen_items = seen_items
        self.candidate_items = np.asarray(list(candidate_items), dtype=np.int64)
        if self.candidate_items.size == 0:
            raise ValueError("candidate_items for negative sampling must be non-empty")

    def __len__(self) -> int:
        return len(self.users)

    def _sample_negative(self, user_idx: int) -> int:
        seen = self.seen_items.get(int(user_idx), set())
        while True:
            neg = int(self.candidate_items[random.randrange(len(self.candidate_items))])
            if neg not in seen:
                return neg

    def __getitem__(self, idx: int):
        u = int(self.users[idx])
        pos_i = int(self.items[idx])
        rating = float(self.ratings[idx])
        neg_i = self._sample_negative(u)
        return {
            "user_idx": u,
            "pos_item_idx": pos_i,
            "neg_item_idx": neg_i,
            "rating": rating,
        }


class SequencePairwiseDataset(Dataset):
    def __init__(
        self,
        train_df: pd.DataFrame,
        candidate_items: Sequence[int],
        positive_threshold: float,
        max_history: int,
        min_history: int,
    ):
        self.samples: List[dict] = []
        grouped = train_df.sort_values(["user_idx", "timestamp", "item_idx"]).groupby("user_idx")
        self.candidate_items = np.asarray(list(candidate_items), dtype=np.int64)
        if self.candidate_items.size == 0:
            raise ValueError("candidate_items for negative sampling must be non-empty")
        self.user_seen: Dict[int, set] = defaultdict(set)

        for user_idx, g in tqdm(grouped, desc="Building sequence samples"):
            history_items: List[int] = []
            history_ratings: List[float] = []
            history_timestamps: List[int] = []
            seen_until_now: set = set()

            for row in g.itertuples(index=False):
                item_idx = int(row.item_idx)
                rating = float(row.rating)
                timestamp = int(row.timestamp)
                if rating >= positive_threshold and len(history_items) >= min_history:
                    self.samples.append(
                        {
                            "user_idx": int(user_idx),
                            "hist_items": history_items[-max_history:].copy(),
                            "hist_ratings": history_ratings[-max_history:].copy(),
                            "hist_timestamps": history_timestamps[-max_history:].copy(),
                            "target_item_idx": item_idx,
                            "target_rating": rating,
                            "target_timestamp": timestamp,
                            "seen_until_now": seen_until_now.copy(),
                        }
                    )
                history_items.append(item_idx)
                history_ratings.append(rating)
                history_timestamps.append(timestamp)
                seen_until_now.add(item_idx)
                self.user_seen[int(user_idx)].add(item_idx)

    def __len__(self) -> int:
        return len(self.samples)

    def _sample_negative(self, seen: set) -> int:
        while True:
            neg = int(self.candidate_items[random.randrange(len(self.candidate_items))])
            if neg not in seen:
                return neg

    def __getitem__(self, idx: int):
        sample = self.samples[idx]
        neg_i = self._sample_negative(sample["seen_until_now"])
        return {
            "user_idx": sample["user_idx"],
            "hist_items": sample["hist_items"],
            "hist_ratings": sample["hist_ratings"],
            "hist_timestamps": sample["hist_timestamps"],
            "target_timestamp": sample["target_timestamp"],
            "pos_item_idx": sample["target_item_idx"],
            "neg_item_idx": neg_i,
            "rating": sample["target_rating"],
        }


class ObservedEdgeDataset(Dataset):
    def __init__(self, train_df: pd.DataFrame, rating2class: Dict[float, int]):
        self.users = train_df["user_idx"].astype(np.int64).to_numpy()
        self.items = train_df["item_idx"].astype(np.int64).to_numpy()
        self.ratings = train_df["rating"].astype(np.float32).to_numpy()
        self.timestamps = train_df["timestamp"].astype(np.int64).to_numpy()
        self.classes = np.asarray([rating2class[float(r)] for r in self.ratings], dtype=np.int64)

    def __len__(self) -> int:
        return len(self.users)

    def __getitem__(self, idx: int):
        return {
            "user_idx": int(self.users[idx]),
            "item_idx": int(self.items[idx]),
            "rating": float(self.ratings[idx]),
            "timestamp": int(self.timestamps[idx]),
            "rating_class": int(self.classes[idx]),
        }


def sequence_collate_fn(batch: List[dict]):
    bsz = len(batch)
    max_len = max(len(x["hist_items"]) for x in batch)
    hist_items = torch.zeros((bsz, max_len), dtype=torch.long)
    hist_ratings = torch.zeros((bsz, max_len), dtype=torch.float32)
    hist_timestamps = torch.zeros((bsz, max_len), dtype=torch.long)
    hist_mask = torch.zeros((bsz, max_len), dtype=torch.bool)
    user_idx = torch.zeros(bsz, dtype=torch.long)
    pos_item_idx = torch.zeros(bsz, dtype=torch.long)
    neg_item_idx = torch.zeros(bsz, dtype=torch.long)
    rating = torch.zeros(bsz, dtype=torch.float32)
    target_timestamp = torch.zeros(bsz, dtype=torch.long)

    for i, sample in enumerate(batch):
        n = len(sample["hist_items"])
        user_idx[i] = sample["user_idx"]
        pos_item_idx[i] = sample["pos_item_idx"]
        neg_item_idx[i] = sample["neg_item_idx"]
        rating[i] = sample["rating"]
        target_timestamp[i] = sample["target_timestamp"]
        hist_items[i, :n] = torch.tensor(sample["hist_items"], dtype=torch.long)
        hist_ratings[i, :n] = torch.tensor(sample["hist_ratings"], dtype=torch.float32)
        hist_timestamps[i, :n] = torch.tensor(sample["hist_timestamps"], dtype=torch.long)
        hist_mask[i, :n] = True

    return {
        "user_idx": user_idx,
        "pos_item_idx": pos_item_idx,
        "neg_item_idx": neg_item_idx,
        "rating": rating,
        "target_timestamp": target_timestamp,
        "hist_items": hist_items,
        "hist_ratings": hist_ratings,
        "hist_timestamps": hist_timestamps,
        "hist_mask": hist_mask,
    }

In [ ]:
from torch.utils.checkpoint import checkpoint

def build_eval_targets(eval_df: pd.DataFrame, cfg_eval: EvalConfig, gain_type: str) -> Dict[int, Dict[int, float]]:
    targets: Dict[int, Dict[int, float]] = defaultdict(dict)
    if eval_df.empty:
        return {}
    grouped = eval_df.groupby("user_idx")
    for user_idx, g in grouped:
        gains = gain_from_rating(
            g["rating"].to_numpy(),
            gain_type=gain_type,
            positive_threshold=cfg_eval.positive_threshold,
            neutral_rating=cfg_eval.neutral_rating,
            power_beta=cfg_eval.power_beta,
            power_gamma=cfg_eval.power_gamma,
        )
        items = g["item_idx"].astype(int).to_numpy()
        tgt = {int(i): float(gain) for i, gain in zip(items, gains) if gain > 0}
        if tgt or not cfg_eval.skip_users_with_no_relevant_targets:
            targets[int(user_idx)] = tgt
    return dict(targets)


def dcg_at_k(gains: Sequence[float], k: int) -> float:
    gains = np.asarray(gains[:k], dtype=np.float32)
    if gains.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, gains.size + 2))
    return float(np.sum(gains * discounts))


def compute_ranking_metrics(
    ranked_items: np.ndarray,
    target_gain_dict: Dict[int, float],
    ks: Sequence[int],
    metric_names: Sequence[str],
) -> Dict[str, float]:
    metrics: Dict[str, float] = {}
    gains_ranked = np.asarray([target_gain_dict.get(int(i), 0.0) for i in ranked_items], dtype=np.float32)
    binary_ranked = (gains_ranked > 0).astype(np.float32)
    relevant_total = int(np.sum(np.asarray(list(target_gain_dict.values()), dtype=np.float32) > 0))
    ideal_gains = sorted(target_gain_dict.values(), reverse=True)

    for k in ks:
        topk_gains = gains_ranked[:k]
        topk_binary = binary_ranked[:k]
        hits = int(np.sum(topk_binary))

        if "hitrate" in metric_names:
            metrics[f"hitrate@{k}"] = float(hits > 0)
        if "precision" in metric_names:
            metrics[f"precision@{k}"] = float(hits / max(k, 1))
        if "recall" in metric_names:
            metrics[f"recall@{k}"] = float(hits / relevant_total) if relevant_total > 0 else 0.0
        if "ndcg" in metric_names:
            dcg = dcg_at_k(topk_gains, k)
            idcg = dcg_at_k(ideal_gains, min(k, len(ideal_gains)))
            metrics[f"ndcg@{k}"] = float(dcg / idcg) if idcg > 0 else 0.0
        if "mrr" in metric_names or "map" in metric_names:
            rel_positions = np.flatnonzero(topk_binary > 0)
            if "mrr" in metric_names:
                metrics[f"mrr@{k}"] = float(1.0 / (rel_positions[0] + 1)) if rel_positions.size > 0 else 0.0
            if "map" in metric_names:
                if rel_positions.size > 0:
                    prefix_hits = np.cumsum(topk_binary)
                    precisions = prefix_hits[rel_positions] / (rel_positions + 1)
                    ap_denom = min(relevant_total, k)
                    metrics[f"map@{k}"] = float(np.sum(precisions) / ap_denom) if ap_denom > 0 else 0.0
                else:
                    metrics[f"map@{k}"] = 0.0
    return metrics


def aggregate_metrics(metric_list: List[Dict[str, float]]) -> Dict[str, float]:
    if not metric_list:
        return {}
    keys = sorted(set().union(*metric_list))
    return {k: float(np.mean([row.get(k, 0.0) for row in metric_list])) for k in keys}


class ItemRepresentationMixin:
    def build_item_base(self, item_idx: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError



class FeatureAwareItemEncoder(nn.Module):
    def __init__(
        self,
        num_items: int,
        feature_tensor: torch.Tensor,
        embedding_dim: int,
        feature_cfg: FeatureConfig,
        trainable_item_mask: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.num_items = int(num_items)
        self.embedding_dim = int(embedding_dim)
        self.feature_cfg = feature_cfg
        self.item_id_emb = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.item_id_emb.weight, std=0.02)

        self.register_buffer("item_features", feature_tensor)
        if trainable_item_mask is None:
            trainable_item_mask = torch.ones(num_items, dtype=torch.float32)
        self.register_buffer("trainable_item_mask", trainable_item_mask.float().view(-1, 1))
        feat_dim = int(feature_tensor.shape[1]) if feature_tensor is not None else 0
        self.use_features = feature_tensor is not None and feature_cfg.use_item_features and feat_dim > 0
        self.feature_dropout = nn.Dropout(feature_cfg.feature_dropout)
        self.init_only_mode = self.use_features and feature_cfg.item_feature_mode == "init_only"

        if self.use_features and not self.init_only_mode:
            self.feature_proj = nn.Linear(feat_dim, embedding_dim)
            nn.init.xavier_uniform_(self.feature_proj.weight)
            nn.init.zeros_(self.feature_proj.bias)
            if feature_cfg.item_feature_mode == "concat_mlp":
                self.concat_mlp = nn.Sequential(
                    nn.Linear(embedding_dim * 2, feature_cfg.feature_hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(feature_cfg.feature_dropout),
                    nn.Linear(feature_cfg.feature_hidden_dim, embedding_dim),
                )
            else:
                self.concat_mlp = None
        else:
            self.feature_proj = None
            self.concat_mlp = None

        if self.init_only_mode:
            init_vectors = self._build_init_only_vectors(feature_tensor)
            with torch.no_grad():
                self.item_id_emb.weight.copy_(init_vectors)

    def _build_init_only_vectors(self, feature_tensor: torch.Tensor) -> torch.Tensor:
        feat_np = feature_tensor.detach().cpu().numpy().astype(np.float32, copy=False)
        if feat_np.ndim != 2 or feat_np.shape[0] == 0:
            return self.item_id_emb.weight.detach().clone()

        if feat_np.shape[1] == self.embedding_dim:
            init_np = feat_np
        else:
            max_components = min(feat_np.shape[1], max(1, feat_np.shape[0] - 1))
            n_components = max(1, min(self.embedding_dim, max_components))
            svd = TruncatedSVD(
                n_components=n_components,
                n_iter=self.feature_cfg.init_only_svd_n_iter,
                random_state=self.feature_cfg.init_only_random_state,
            )
            init_np = svd.fit_transform(feat_np).astype(np.float32)
            if n_components < self.embedding_dim:
                pad = np.zeros((init_np.shape[0], self.embedding_dim - n_components), dtype=np.float32)
                init_np = np.concatenate([init_np, pad], axis=1)

        init_tensor = torch.tensor(init_np, dtype=torch.float32)
        init_std = float(init_tensor.std().item()) if init_tensor.numel() > 0 else 0.0
        if init_std > 1e-8:
            init_tensor = init_tensor / init_std * 0.02
        return init_tensor

    def forward(self, item_idx: torch.Tensor) -> torch.Tensor:
        raw_item_id_vec = self.item_id_emb(item_idx)
        if self.init_only_mode:
            return raw_item_id_vec

        item_id_vec = raw_item_id_vec * self.trainable_item_mask[item_idx]
        if not self.use_features:
            return item_id_vec

        feat_vec = self.feature_proj(self.feature_dropout(self.item_features[item_idx]))
        mode = self.feature_cfg.item_feature_mode
        if mode == "none":
            return item_id_vec
        if mode == "add":
            return item_id_vec + feat_vec
        if mode == "project_only":
            return feat_vec
        if mode == "concat_mlp":
            return self.concat_mlp(torch.cat([item_id_vec, feat_vec], dim=-1))
        raise ValueError(f"Unknown item_feature_mode={mode}")

    def all_item_vectors(self) -> torch.Tensor:
        idx = torch.arange(self.num_items, device=self.item_id_emb.weight.device)
        return self.forward(idx)


class MFModel(nn.Module):
    def __init__(
        self,
        num_users: int,
        num_items: int,
        feature_tensor: torch.Tensor,
        feature_cfg: FeatureConfig,
        mf_cfg: MFConfig,
        trainable_item_mask: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, mf_cfg.embedding_dim)
        nn.init.normal_(self.user_emb.weight, std=0.02)
        self.item_encoder = FeatureAwareItemEncoder(num_items, feature_tensor, mf_cfg.embedding_dim, feature_cfg, trainable_item_mask)
        self.dropout = nn.Dropout(mf_cfg.dropout)

    def score(self, user_idx: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:
        u = self.dropout(self.user_emb(user_idx))
        v = self.dropout(self.item_encoder(item_idx))
        return (u * v).sum(dim=-1)

    def forward(self, user_idx: torch.Tensor, pos_item_idx: torch.Tensor, neg_item_idx: Optional[torch.Tensor] = None):
        pos_scores = self.score(user_idx, pos_item_idx)
        if neg_item_idx is None:
            return pos_scores
        neg_scores = self.score(user_idx, neg_item_idx)
        return pos_scores, neg_scores

    def score_all_items(self, user_idx: torch.Tensor) -> torch.Tensor:
        u = self.user_emb(user_idx)
        all_items = self.item_encoder.all_item_vectors()
        return u @ all_items.T


class AttentionPooling(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.proj = nn.Linear(dim, 1)

    def forward(self, x: torch.Tensor, mask: torch.Tensor, weights: Optional[torch.Tensor] = None) -> torch.Tensor:
        logits = self.proj(x).squeeze(-1)
        if weights is not None:
            logits = logits + torch.log(torch.clamp(weights, min=1e-8))
        logits = logits.masked_fill(~mask, -1e9)
        attn = torch.softmax(logits, dim=-1)
        return torch.bmm(attn.unsqueeze(1), x).squeeze(1)


class TwoTowerModel(nn.Module):
    def __init__(
        self,
        num_users: int,
        num_items: int,
        feature_tensor: torch.Tensor,
        feature_cfg: FeatureConfig,
        model_cfg: TwoTowerConfig,
        trainable_item_mask: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.model_cfg = model_cfg
        self.embedding_dim = model_cfg.embedding_dim
        self.num_users = int(num_users)
        self.item_encoder = FeatureAwareItemEncoder(num_items, feature_tensor, model_cfg.embedding_dim, feature_cfg, trainable_item_mask)
        self.use_user_id = model_cfg.use_user_id_embedding
        if self.use_user_id:
            self.user_id_emb = nn.Embedding(num_users, model_cfg.embedding_dim)
            nn.init.normal_(self.user_id_emb.weight, std=0.02)
        else:
            self.user_id_emb = None
        self.rating_proj = nn.Linear(1, model_cfg.embedding_dim)
        self.user_mlp = nn.Sequential(
            nn.Linear(model_cfg.embedding_dim * (2 if self.use_user_id else 1), model_cfg.hidden_dim),
            nn.ReLU(),
            nn.Dropout(model_cfg.dropout),
            nn.Linear(model_cfg.hidden_dim, model_cfg.embedding_dim),
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(model_cfg.embedding_dim, model_cfg.hidden_dim),
            nn.ReLU(),
            nn.Dropout(model_cfg.dropout),
            nn.Linear(model_cfg.hidden_dim, model_cfg.embedding_dim),
        )
        self.attention_pool = AttentionPooling(model_cfg.embedding_dim)
        self.dropout = nn.Dropout(model_cfg.dropout)

    def encode_history(
        self,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        hist_vecs = self.item_encoder(hist_items)
        if self.model_cfg.use_rating_in_history:
            hist_vecs = hist_vecs + self.rating_proj(hist_ratings.unsqueeze(-1))
        weights = None
        if self.model_cfg.history_pooling in {"weighted_mean", "attention"}:
            weights = torch.ones_like(hist_ratings)
            if self.model_cfg.use_rating_in_history:
                weights = weights * rating_weight(hist_ratings, self.model_cfg.rating_weight_fn)
            if self.model_cfg.use_time_in_history and target_timestamp is not None:
                delta = (target_timestamp.unsqueeze(1) - hist_timestamps).clamp(min=0)
                weights = weights * time_weight(delta.float(), self.model_cfg.time_weight_fn, self.model_cfg.time_weight_gamma)
            weights = weights * hist_mask.float()

        if self.model_cfg.history_pooling == "mean":
            mask_f = hist_mask.float().unsqueeze(-1)
            pooled = (hist_vecs * mask_f).sum(dim=1) / torch.clamp(mask_f.sum(dim=1), min=1.0)
        elif self.model_cfg.history_pooling == "weighted_mean":
            w = weights.unsqueeze(-1)
            pooled = (hist_vecs * w).sum(dim=1) / torch.clamp(w.sum(dim=1), min=1e-8)
        elif self.model_cfg.history_pooling == "attention":
            pooled = self.attention_pool(hist_vecs, hist_mask, weights)
        else:
            raise ValueError(f"Unknown history_pooling={self.model_cfg.history_pooling}")
        return pooled

    def encode_user(
        self,
        user_idx: torch.Tensor,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        parts = []
        hist_vec = self.encode_history(hist_items, hist_ratings, hist_timestamps, hist_mask, target_timestamp)
        parts.append(hist_vec)
        if self.use_user_id:
            valid_user = ((user_idx >= 0) & (user_idx < self.num_users)).float().unsqueeze(-1)
            safe_user_idx = torch.where(user_idx >= 0, torch.clamp(user_idx, max=self.num_users - 1), torch.zeros_like(user_idx))
            parts.append(self.user_id_emb(safe_user_idx) * valid_user)
        user_vec = torch.cat(parts, dim=-1)
        return self.user_mlp(self.dropout(user_vec))

    def encode_item(self, item_idx: torch.Tensor) -> torch.Tensor:
        return self.item_mlp(self.dropout(self.item_encoder(item_idx)))

    def score_batch(self, batch: dict) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.forward(
            batch["user_idx"],
            batch["pos_item_idx"],
            batch["neg_item_idx"],
            batch["hist_items"],
            batch["hist_ratings"],
            batch["hist_timestamps"],
            batch["hist_mask"],
            batch["target_timestamp"],
        )

    def forward(
        self,
        user_idx: torch.Tensor,
        pos_item_idx: torch.Tensor,
        neg_item_idx: torch.Tensor,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        user_vec = self.encode_user(
            user_idx,
            hist_items,
            hist_ratings,
            hist_timestamps,
            hist_mask,
            target_timestamp,
        )
        pos_vec = self.encode_item(pos_item_idx)
        neg_vec = self.encode_item(neg_item_idx)
        pos_scores = (user_vec * pos_vec).sum(dim=-1)
        neg_scores = (user_vec * neg_vec).sum(dim=-1)
        return pos_scores, neg_scores

    def score_all_items_for_histories(self, user_vecs: torch.Tensor) -> torch.Tensor:
        item_vecs = self.encode_item(torch.arange(self.item_encoder.num_items, device=user_vecs.device))
        return user_vecs @ item_vecs.T


class RelationGCMLayer(nn.Module):
    def __init__(self, dim_in: int, dim_out: int, num_relations: int, dropout: float):
        super().__init__()
        self.user_rel_linears = nn.ModuleList([nn.Linear(dim_in, dim_out, bias=False) for _ in range(num_relations)])
        self.item_rel_linears = nn.ModuleList([nn.Linear(dim_in, dim_out, bias=False) for _ in range(num_relations)])
        self.user_self = nn.Linear(dim_in, dim_out, bias=True)
        self.item_self = nn.Linear(dim_in, dim_out, bias=True)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        user_h: torch.Tensor,
        item_h: torch.Tensor,
        adjs_u_from_i: Sequence[torch.Tensor],
        adjs_i_from_u: Sequence[torch.Tensor],
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        user_msg = self.user_self(user_h)
        item_msg = self.item_self(item_h)
        for lin_u, lin_i, a_ui, a_iu in zip(self.user_rel_linears, self.item_rel_linears, adjs_u_from_i, adjs_i_from_u):
            user_msg = user_msg + lin_u(torch.sparse.mm(a_ui, item_h))
            item_msg = item_msg + lin_i(torch.sparse.mm(a_iu, user_h))
        user_out = F.relu(self.dropout(user_msg))
        item_out = F.relu(self.dropout(item_msg))
        return user_out, item_out


class BilinearRatingDecoder(nn.Module):
    def __init__(self, dim: int, num_relations: int):
        super().__init__()
        self.Q = nn.Parameter(torch.empty(num_relations, dim, dim))
        nn.init.xavier_uniform_(self.Q)

    def forward(self, user_vec: torch.Tensor, item_vec: torch.Tensor) -> torch.Tensor:
        # logits: [B, R]
        logits = torch.einsum("bd,rdh,bh->br", user_vec, self.Q, item_vec)
        return logits


class RelationGCMCModel(nn.Module):
    def __init__(
        self,
        num_users: int,
        num_items: int,
        feature_tensor: torch.Tensor,
        feature_cfg: FeatureConfig,
        graph_cfg: GraphConfig,
        rating_values: List[float],
        trainable_item_mask: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.graph_cfg = graph_cfg
        self.rating_values = [float(x) for x in rating_values]
        self.num_relations = len(rating_values)
        self.user_emb = nn.Embedding(num_users, graph_cfg.embedding_dim)
        nn.init.normal_(self.user_emb.weight, std=0.02)
        self.item_encoder = FeatureAwareItemEncoder(num_items, feature_tensor, graph_cfg.embedding_dim, feature_cfg, trainable_item_mask)
        self.layers = nn.ModuleList(
            [
                RelationGCMLayer(graph_cfg.embedding_dim, graph_cfg.embedding_dim, self.num_relations, graph_cfg.dropout)
                for _ in range(graph_cfg.num_layers)
            ]
        )
        self.decoder = BilinearRatingDecoder(graph_cfg.embedding_dim, self.num_relations)
        self._cached_graph = None

    def set_graph(self, graph_dict: dict) -> None:
        self._cached_graph = graph_dict

    def encode_full_graph(self) -> Tuple[torch.Tensor, torch.Tensor]:
        if self._cached_graph is None:
            raise RuntimeError("Graph is not set. Call model.set_graph(graph_dict) first.")
        adjs_u_from_i = self._cached_graph["adjs_u_from_i"]
        adjs_i_from_u = self._cached_graph["adjs_i_from_u"]
        user_h = self.user_emb.weight
        item_h = self.item_encoder.all_item_vectors()

        user_sum = user_h
        item_sum = item_h
        num_states = 1

        use_checkpoint = bool(
            getattr(self.graph_cfg, "checkpoint_encoder_layers", False)
            and self.training
            and torch.is_grad_enabled()
        )

        for layer in self.layers:
            if use_checkpoint:
                user_h, item_h = checkpoint(
                    lambda uh, ih, layer=layer: layer(uh, ih, adjs_u_from_i, adjs_i_from_u),
                    user_h,
                    item_h,
                    use_reentrant=False,
                )
            else:
                user_h, item_h = layer(user_h, item_h, adjs_u_from_i, adjs_i_from_u)
            user_sum = user_sum + user_h
            item_sum = item_sum + item_h
            num_states += 1

        user_z = user_sum / num_states
        item_z = item_sum / num_states
        return user_z, item_z

    def decode(self, user_vec: torch.Tensor, item_vec: torch.Tensor) -> torch.Tensor:
        return self.decoder(user_vec, item_vec)

    def _score_weights(self, device: torch.device, dtype: torch.dtype, positive_threshold: float = 4.0) -> torch.Tensor:
        relation_values = torch.tensor(self.rating_values, device=device, dtype=dtype)
        if self.graph_cfg.relation_score_mode == "prob_positive":
            return (relation_values >= positive_threshold).float().to(dtype=dtype)
        return relation_values

    def score_from_logits(self, logits: torch.Tensor, positive_threshold: float = 4.0) -> torch.Tensor:
        probs = torch.softmax(logits, dim=-1)
        weights = self._score_weights(logits.device, logits.dtype, positive_threshold)
        return probs @ weights

    def score_user_vectors_all_items(
        self,
        user_vecs: torch.Tensor,
        item_vecs: torch.Tensor,
        positive_threshold: float = 4.0,
        item_batch_size: Optional[int] = None,
    ) -> torch.Tensor:
        projected_users = torch.einsum("bd,rdh->brh", user_vecs, self.decoder.Q)
        weights = self._score_weights(user_vecs.device, user_vecs.dtype, positive_threshold)

        if item_batch_size is None or item_batch_size <= 0 or item_batch_size >= item_vecs.shape[0]:
            logits = torch.einsum("brh,nh->bnr", projected_users, item_vecs)
            probs = torch.softmax(logits, dim=-1)
            return torch.einsum("bnr,r->bn", probs, weights)

        chunks = []
        for start in range(0, item_vecs.shape[0], int(item_batch_size)):
            item_chunk = item_vecs[start:start + int(item_batch_size)]
            logits = torch.einsum("brh,nh->bnr", projected_users, item_chunk)
            probs = torch.softmax(logits, dim=-1)
            chunk_scores = torch.einsum("bnr,r->bn", probs, weights)
            chunks.append(chunk_scores)
        return torch.cat(chunks, dim=1)

    def score_all_items(self, user_indices: torch.Tensor) -> torch.Tensor:
        user_z, item_z = self.encode_full_graph()
        return self.score_user_vectors_all_items(user_z[user_indices], item_z, positive_threshold=4.0)


class SupportAwareGCMCModel(RelationGCMCModel):
    def __init__(
        self,
        num_users: int,
        num_items: int,
        feature_tensor: torch.Tensor,
        feature_cfg: FeatureConfig,
        graph_cfg: GraphConfig,
        rating_values: List[float],
        trainable_item_mask: Optional[torch.Tensor] = None,
    ):
        super().__init__(
            num_users=num_users,
            num_items=num_items,
            feature_tensor=feature_tensor,
            feature_cfg=feature_cfg,
            graph_cfg=graph_cfg,
            rating_values=rating_values,
            trainable_item_mask=trainable_item_mask,
        )
        self.relation_emb = nn.Embedding(self.num_relations, graph_cfg.embedding_dim)
        nn.init.normal_(self.relation_emb.weight, std=0.02)
        self.support_attention_pool = AttentionPooling(graph_cfg.embedding_dim)
        self.support_user_mlp = nn.Sequential(
            nn.Linear(graph_cfg.embedding_dim, graph_cfg.hidden_dim),
            nn.ReLU(),
            nn.Dropout(graph_cfg.dropout),
            nn.Linear(graph_cfg.hidden_dim, graph_cfg.embedding_dim),
        )

    def ratings_to_relation_ids(self, ratings: torch.Tensor) -> torch.Tensor:
        relation_values = torch.tensor(self.rating_values, device=ratings.device, dtype=ratings.dtype)
        rel_ids = torch.argmin(torch.abs(ratings.unsqueeze(-1) - relation_values.view(*([1] * ratings.dim()), -1)), dim=-1)
        return rel_ids.long()

    def encode_support_users(
        self,
        hist_items: torch.Tensor,
        hist_ratings: torch.Tensor,
        hist_timestamps: torch.Tensor,
        hist_mask: torch.Tensor,
        target_timestamp: Optional[torch.Tensor] = None,
        item_vectors: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if item_vectors is None:
            _, item_vectors = self.encode_full_graph()
        hist_vecs = item_vectors[hist_items]
        rel_ids = self.ratings_to_relation_ids(hist_ratings)
        hist_vecs = hist_vecs + self.relation_emb(rel_ids)

        weights = hist_mask.float()
        if self.graph_cfg.support_use_rating:
            weights = weights * rating_weight(hist_ratings, self.graph_cfg.rating_weight_fn)
        if self.graph_cfg.support_use_time and target_timestamp is not None:
            delta = (target_timestamp.unsqueeze(1) - hist_timestamps).clamp(min=0)
            weights = weights * time_weight(delta.float(), self.graph_cfg.support_time_weight_fn, self.graph_cfg.support_time_weight_gamma)

        pooling = self.graph_cfg.support_history_pooling
        if pooling == "attention":
            pooled = self.support_attention_pool(hist_vecs, hist_mask, weights)
        else:
            denom = hist_mask.float().sum(dim=1, keepdim=True).clamp(min=1.0) if pooling == "mean" else weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
            pooled = (hist_vecs * weights.unsqueeze(-1)).sum(dim=1) / denom
        pooled = pooled * hist_mask.any(dim=1, keepdim=True).float()
        return self.support_user_mlp(pooled)

# ============================================================
# Method 2: Ordinal NGCF
# ============================================================
# Рейтинг используется как вес ребра в одном user-item графе:
#   edge_weight = w(r_uv)
#
# Поддерживаются три функции веса:
#   1) linear_norm       — фиксированная ordinal-функция
#   2) exp_mixture       — обучаемая гладкая монотонная функция
#   3) softplus_spline   — обучаемая гибкая spline-функция


class ExpMixtureRatingWeight(nn.Module):
    """
    Smooth learnable monotone rating weight function.

    x = normalized rating in [0, 1]

    w(x) = eps + (1 - eps) * sum_m pi_m * phi_{lambda_m}(x)
    phi_lam(x) = (exp(lam * x) - 1) / (exp(lam) - 1)

    Negative lambdas produce saturating curves.
    Positive lambdas produce threshold-like curves.
    A softmax mixture keeps the function monotone and bounded.
    """
    def __init__(
        self,
        r_min: float = 0.5,
        r_max: float = 5.0,
        num_terms: int = 4,
        eps: float = 1e-3,
        lambda_clip: float = 8.0,
    ):
        super().__init__()
        self.r_min = float(r_min)
        self.r_max = float(r_max)
        self.num_terms = int(num_terms)
        self.eps = float(eps)
        self.lambda_clip = float(lambda_clip)

        self.logits = nn.Parameter(torch.zeros(self.num_terms))
        init_lambdas = torch.linspace(-3.0, 3.0, self.num_terms)
        self.raw_lambdas = nn.Parameter(init_lambdas)

    def _normalize_rating(self, rating: torch.Tensor) -> torch.Tensor:
        denom = max(self.r_max - self.r_min, 1e-8)
        return torch.clamp((rating.float() - self.r_min) / denom, min=0.0, max=1.0)

    def _basis(self, x: torch.Tensor, lambdas: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(-1)
        lambdas = lambdas.view(1, -1)

        denom = torch.expm1(lambdas)
        denom_safe = torch.where(denom.abs() < 1e-6, torch.ones_like(denom), denom)
        phi = torch.expm1(lambdas * x) / denom_safe
        phi = torch.where(lambdas.abs() < 1e-6, x.expand_as(phi), phi)

        return torch.clamp(phi, min=0.0, max=1.0)

    def forward(self, rating: torch.Tensor) -> torch.Tensor:
        x = self._normalize_rating(rating)
        pi = F.softmax(self.logits, dim=0)
        lambdas = torch.clamp(self.raw_lambdas, min=-self.lambda_clip, max=self.lambda_clip)
        phi = self._basis(x, lambdas)
        w = (phi * pi.view(1, -1)).sum(dim=-1)
        return self.eps + (1.0 - self.eps) * w


class SoftplusSplineRatingWeight(nn.Module):
    """
    Flexible learnable monotone rating weight function.

    x = normalized rating in [0, 1]

    g(x) = offset + a0 * x + sum_m a_m * [
        softplus(k * (x - t_m)) - softplus(k * (0 - t_m))
    ]

    a0 >= 0, a_m >= 0, k > 0
    => g(x) is monotone non-decreasing.

    Final weight:
        w(x) = eps + (1 - eps) * g(x) / g(1)
    """
    def __init__(
        self,
        r_min: float = 0.5,
        r_max: float = 5.0,
        eps: float = 1e-3,
        num_knots: int = 5,
        init_kappa: float = 8.0,
        learn_kappa: bool = True,
    ):
        super().__init__()
        self.r_min = float(r_min)
        self.r_max = float(r_max)
        self.eps = float(eps)
        self.num_knots = int(num_knots)
        self.learn_kappa = bool(learn_kappa)

        knots = torch.linspace(0.0, 1.0, self.num_knots + 2)[1:-1]
        self.register_buffer("knots", knots, persistent=False)

        # Small positive offset, positive linear slope, positive hinge coefficients.
        self.raw_offset = nn.Parameter(torch.tensor(-5.0))
        self.raw_base_slope = nn.Parameter(torch.tensor(0.0))
        self.raw_coeffs = nn.Parameter(torch.zeros(self.num_knots))

        if self.learn_kappa:
            # softplus(8) ≈ 8, so this initializes near the requested sharpness.
            self.raw_kappa = nn.Parameter(torch.tensor(float(init_kappa)))
        else:
            self.register_buffer("fixed_kappa", torch.tensor(float(init_kappa)), persistent=False)

    def _normalize_rating(self, rating: torch.Tensor) -> torch.Tensor:
        denom = max(self.r_max - self.r_min, 1e-8)
        return torch.clamp((rating.float() - self.r_min) / denom, min=0.0, max=1.0)

    def _get_kappa(self) -> torch.Tensor:
        if self.learn_kappa:
            return F.softplus(self.raw_kappa) + 1e-3
        return self.fixed_kappa

    def _basis(self, x: torch.Tensor, kappa: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(-1)
        knots = self.knots.to(device=x.device, dtype=x.dtype).view(1, -1)
        value = F.softplus(kappa * (x - knots))
        value_at_zero = F.softplus(kappa * (0.0 - knots))
        return value - value_at_zero

    def _raw_g(self, x: torch.Tensor) -> torch.Tensor:
        kappa = self._get_kappa()
        offset = F.softplus(self.raw_offset)
        base_slope = F.softplus(self.raw_base_slope)
        coeffs = F.softplus(self.raw_coeffs)

        basis = self._basis(x, kappa)
        return offset + base_slope * x + (basis * coeffs.view(1, -1)).sum(dim=-1)

    def forward(self, rating: torch.Tensor) -> torch.Tensor:
        x = self._normalize_rating(rating)
        g = self._raw_g(x)

        x_one = torch.ones(1, device=rating.device, dtype=rating.float().dtype)
        g_one = self._raw_g(x_one).clamp_min(1e-8)

        w = g / g_one
        return self.eps + (1.0 - self.eps) * w


class RatingEdgeWeight(nn.Module):
    """
    Unified wrapper around rating-edge weight functions for Ordinal NGCF.

    kind:
      - linear_norm
      - exp_mixture
      - softplus_spline
    """
    def __init__(self, graph_cfg: GraphConfig, rating_values: Sequence[float]):
        super().__init__()
        self.kind = graph_cfg.rating_weight_fn
        self.r_min = float(graph_cfg.rating_r_min)
        self.r_max = float(graph_cfg.rating_r_max)
        self.eps = float(graph_cfg.rating_weight_eps)

        rating_values = sorted([float(x) for x in rating_values])
        self.register_buffer(
            "rating_values",
            torch.tensor(rating_values, dtype=torch.float32),
            persistent=False,
        )

        if self.kind == "exp_mixture":
            self.fn = ExpMixtureRatingWeight(
                r_min=graph_cfg.rating_r_min,
                r_max=graph_cfg.rating_r_max,
                num_terms=graph_cfg.rating_exp_terms,
                eps=graph_cfg.rating_weight_eps,
                lambda_clip=graph_cfg.rating_exp_lambda_clip,
            )
        elif self.kind == "softplus_spline":
            self.fn = SoftplusSplineRatingWeight(
                r_min=graph_cfg.rating_r_min,
                r_max=graph_cfg.rating_r_max,
                eps=graph_cfg.rating_weight_eps,
                num_knots=graph_cfg.rating_softplus_num_knots,
                init_kappa=graph_cfg.rating_softplus_init_kappa,
                learn_kappa=graph_cfg.rating_softplus_learn_kappa,
            )
        elif self.kind == "linear_norm":
            self.fn = None
        else:
            raise ValueError(
                "Ordinal NGCF supports rating_weight_fn in "
                "{'linear_norm', 'exp_mixture', 'softplus_spline'}, "
                f"got {self.kind!r}"
            )

    def _normalize_rating(self, rating: torch.Tensor) -> torch.Tensor:
        denom = max(self.r_max - self.r_min, 1e-8)
        return torch.clamp((rating.float() - self.r_min) / denom, min=0.0, max=1.0)

    def forward(self, rating: torch.Tensor) -> torch.Tensor:
        if self.kind == "linear_norm":
            x = self._normalize_rating(rating)
            return self.eps + (1.0 - self.eps) * x
        return self.fn(rating)

    @torch.no_grad()
    def current_table(self, rating_values: Optional[Sequence[float]] = None) -> Dict[float, float]:
        if rating_values is None:
            rating_tensor = self.rating_values.detach().clone()
        else:
            rating_tensor = torch.tensor([float(x) for x in rating_values], dtype=torch.float32)

        device = self.rating_values.device
        rating_tensor = rating_tensor.to(device=device)
        weights = self.forward(rating_tensor)

        return {
            float(r): float(w)
            for r, w in zip(rating_tensor.detach().cpu(), weights.detach().cpu())
        }


class WeightedNGCFLayer(nn.Module):
    """
    NGCF-style graph convolution with weighted bipartite propagation.

    neighbor_u = A_ui @ item_h
    out_u = W1(neighbor_u) + W2(user_h * neighbor_u)

    This is not LightGCN: the layer has transformations, nonlinearity,
    dropout and a bi-interaction term.
    """
    def __init__(self, dim: int, dropout: float, use_l2_norm: bool = True):
        super().__init__()
        self.W1 = nn.Linear(dim, dim, bias=False)
        self.W2 = nn.Linear(dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.LeakyReLU(negative_slope=0.2)
        self.use_l2_norm = bool(use_l2_norm)

        nn.init.xavier_uniform_(self.W1.weight)
        nn.init.xavier_uniform_(self.W2.weight)

    def forward(
        self,
        user_h: torch.Tensor,
        item_h: torch.Tensor,
        a_ui: torch.Tensor,
        a_iu: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        user_neigh = torch.sparse.mm(a_ui, item_h)
        item_neigh = torch.sparse.mm(a_iu, user_h)

        user_out = self.W1(user_neigh) + self.W2(user_h * user_neigh)
        item_out = self.W1(item_neigh) + self.W2(item_h * item_neigh)

        user_out = self.dropout(self.act(user_out))
        item_out = self.dropout(self.act(item_out))

        if self.use_l2_norm:
            user_out = F.normalize(user_out, p=2, dim=-1)
            item_out = F.normalize(item_out, p=2, dim=-1)

        return user_out, item_out


class OrdinalNGCFModel(nn.Module):
    """
    Ordinal explicit GNN / method 2, implemented as weighted NGCF.

    - one user-item graph;
    - rating is converted into edge weight w(r_uv);
    - item features are used only as initial item-embedding values when
      cfg.features.item_feature_mode == "init_only";
    - cold users are represented by one fixed shared vector, without a
      separate support-history encoder;
    - encoder is NGCF-style weighted propagation;
    - recommendation score is dot product z_u^T z_i;
    - training objective is BPR over positive interactions.
    """
    def __init__(
        self,
        num_users: int,
        num_items: int,
        feature_tensor: torch.Tensor,
        feature_cfg: FeatureConfig,
        graph_cfg: GraphConfig,
        rating_values: Sequence[float],
        trainable_item_mask: Optional[torch.Tensor] = None,
        num_warm_users: Optional[int] = None,
    ):
        super().__init__()
        self.num_users = int(num_users)
        self.num_warm_users = int(num_warm_users) if num_warm_users is not None else int(num_users)
        self.num_items = int(num_items)
        self.graph_cfg = graph_cfg

        if self.num_warm_users <= 0:
            raise ValueError("num_warm_users must be positive")
        if self.num_warm_users > self.num_users:
            raise ValueError("num_warm_users cannot be greater than num_users")

        # Only warm users receive learnable ID embeddings.
        # Cold users are represented by self.cold_user_init_vector, shared by all new users.
        self.user_emb = nn.Embedding(self.num_warm_users, graph_cfg.embedding_dim)
        nn.init.normal_(self.user_emb.weight, std=0.02)

        self.register_buffer(
            "cold_user_init_vector",
            torch.zeros(1, graph_cfg.embedding_dim, dtype=torch.float32),
            persistent=False,
        )

        self.item_encoder = FeatureAwareItemEncoder(
            num_items=num_items,
            feature_tensor=feature_tensor,
            embedding_dim=graph_cfg.embedding_dim,
            feature_cfg=feature_cfg,
            trainable_item_mask=trainable_item_mask,
        )

        self.edge_weight_fn = RatingEdgeWeight(graph_cfg, rating_values)

        self.layers = nn.ModuleList([
            WeightedNGCFLayer(
                dim=graph_cfg.embedding_dim,
                dropout=graph_cfg.dropout,
                use_l2_norm=graph_cfg.ngcf_use_l2_norm,
            )
            for _ in range(graph_cfg.num_layers)
        ])

    @property
    def has_cold_users(self) -> bool:
        return self.num_users > self.num_warm_users

    def _set_buffer(self, name: str, tensor: torch.Tensor) -> None:
        if name in self._buffers:
            self._buffers[name] = tensor
        else:
            self.register_buffer(name, tensor, persistent=False)

    def set_graph_edges(self, train_df: pd.DataFrame) -> None:
        device = next(self.parameters()).device

        edge_user = torch.tensor(
            train_df["user_idx"].astype(np.int64).to_numpy(),
            dtype=torch.long,
            device=device,
        )
        edge_item = torch.tensor(
            train_df["item_idx"].astype(np.int64).to_numpy(),
            dtype=torch.long,
            device=device,
        )
        edge_rating = torch.tensor(
            train_df["rating"].astype(np.float32).to_numpy(),
            dtype=torch.float32,
            device=device,
        )

        if edge_user.numel() > 0 and int(edge_user.max().item()) >= self.num_warm_users:
            raise ValueError(
                "OrdinalNGCFModel expects train graph to contain only warm users. "
                f"Found max user_idx={int(edge_user.max().item())}, "
                f"num_warm_users={self.num_warm_users}."
            )

        if "timestamp" in train_df.columns:
            edge_timestamp = torch.tensor(
                train_df["timestamp"].astype(np.int64).to_numpy(),
                dtype=torch.long,
                device=device,
            )
            latest_ts = int(train_df["timestamp"].max())
        else:
            edge_timestamp = torch.zeros_like(edge_user)
            latest_ts = 0

        self._set_buffer("edge_user", edge_user)
        self._set_buffer("edge_item", edge_item)
        self._set_buffer("edge_rating", edge_rating)
        self._set_buffer("edge_timestamp", edge_timestamp)
        self._set_buffer("latest_timestamp", torch.tensor(latest_ts, dtype=torch.long, device=device))

    @torch.no_grad()
    def initialize_fixed_cold_user_vector(
        self,
        train_df: pd.DataFrame,
        positive_threshold: float = 4.0,
    ) -> None:
        """
        Initializes the shared vector for all cold users.

        This is intentionally not a support encoder: every unseen user receives
        the same fixed vector. Personalization for cold users can only come from
        filtering already-seen support items during evaluation.
        """
        if not self.has_cold_users:
            return

        strategy = getattr(self.graph_cfg, "cold_user_init_strategy", "positive_item_mean")
        device = next(self.parameters()).device
        dtype = self.user_emb.weight.dtype

        if strategy == "zero":
            cold_vec = torch.zeros((1, self.graph_cfg.embedding_dim), device=device, dtype=dtype)

        elif strategy == "mean_warm_user":
            cold_vec = self.user_emb.weight.detach().mean(dim=0, keepdim=True)

        elif strategy == "positive_item_mean":
            if train_df.empty:
                cold_vec = self.user_emb.weight.detach().mean(dim=0, keepdim=True)
            else:
                pos_df = train_df[train_df["rating"] >= float(positive_threshold)]
                if pos_df.empty:
                    pos_df = train_df

                item_idx = torch.tensor(
                    pos_df["item_idx"].astype(np.int64).to_numpy(),
                    dtype=torch.long,
                    device=device,
                )
                ratings = torch.tensor(
                    pos_df["rating"].astype(np.float32).to_numpy(),
                    dtype=torch.float32,
                    device=device,
                )

                item_vecs = self.item_encoder.all_item_vectors().detach()
                selected = item_vecs[item_idx]

                weights = self.edge_weight_fn(ratings).detach().clamp_min(1e-8).view(-1, 1)
                cold_vec = (selected * weights).sum(dim=0, keepdim=True) / weights.sum().clamp_min(1e-8)

                # Keep the cold vector on a comparable scale to warm user embeddings.
                warm_std = self.user_emb.weight.detach().std().clamp_min(1e-8)
                cold_std = cold_vec.detach().std().clamp_min(1e-8)
                cold_vec = cold_vec / cold_std * warm_std

        else:
            raise ValueError(
                "Unknown cold_user_init_strategy="
                f"{strategy}. Expected: positive_item_mean | mean_warm_user | zero"
            )

        self.cold_user_init_vector.copy_(cold_vec.to(device=device, dtype=self.cold_user_init_vector.dtype))

    def initial_user_vectors(self) -> torch.Tensor:
        warm_user_h = self.user_emb.weight
        if not self.has_cold_users:
            return warm_user_h

        num_cold = self.num_users - self.num_warm_users
        cold_h = self.cold_user_init_vector.to(
            device=warm_user_h.device,
            dtype=warm_user_h.dtype,
        ).expand(num_cold, -1)

        return torch.cat([warm_user_h, cold_h], dim=0)

    def build_weighted_adjs(self) -> Tuple[torch.Tensor, torch.Tensor]:
        if not hasattr(self, "edge_user"):
            raise RuntimeError("Graph edges are not set. Call model.set_graph_edges(train_df).")

        u = self.edge_user
        i = self.edge_item

        w = self.edge_weight_fn(self.edge_rating)

        if self.graph_cfg.relation_apply_time_decay:
            delta = (self.latest_timestamp - self.edge_timestamp).clamp(min=0).float()
            w = w * time_weight(
                delta,
                self.graph_cfg.time_weight_fn,
                self.graph_cfg.time_weight_gamma,
            )

        user_deg = torch.zeros(self.num_users, device=w.device, dtype=w.dtype)
        item_deg = torch.zeros(self.num_items, device=w.device, dtype=w.dtype)

        user_deg.scatter_add_(0, u, w)
        item_deg.scatter_add_(0, i, w)

        norm = w * torch.rsqrt(user_deg[u].clamp_min(1e-8) * item_deg[i].clamp_min(1e-8))

        ui_idx = torch.stack([u, i], dim=0)
        iu_idx = torch.stack([i, u], dim=0)

        a_ui = torch.sparse_coo_tensor(
            ui_idx,
            norm,
            size=(self.num_users, self.num_items),
            device=w.device,
        ).coalesce()

        a_iu = torch.sparse_coo_tensor(
            iu_idx,
            norm,
            size=(self.num_items, self.num_users),
            device=w.device,
        ).coalesce()

        return a_ui, a_iu

    def encode_full_graph(self) -> Tuple[torch.Tensor, torch.Tensor]:
        a_ui, a_iu = self.build_weighted_adjs()

        user_h = self.initial_user_vectors()
        item_h = self.item_encoder.all_item_vectors()

        user_states = [user_h]
        item_states = [item_h]

        for layer in self.layers:
            user_h, item_h = layer(user_h, item_h, a_ui, a_iu)
            user_states.append(user_h)
            item_states.append(item_h)

        if self.graph_cfg.ngcf_layer_pooling == "concat":
            user_z = torch.cat(user_states, dim=-1)
            item_z = torch.cat(item_states, dim=-1)
        elif self.graph_cfg.ngcf_layer_pooling == "mean":
            user_z = torch.stack(user_states, dim=0).mean(dim=0)
            item_z = torch.stack(item_states, dim=0).mean(dim=0)
        else:
            raise ValueError(f"Unknown ngcf_layer_pooling={self.graph_cfg.ngcf_layer_pooling}")

        return user_z, item_z

    def score(self, user_idx: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:
        user_z, item_z = self.encode_full_graph()
        return (user_z[user_idx] * item_z[item_idx]).sum(dim=-1)

    def forward(
        self,
        user_idx: torch.Tensor,
        pos_item_idx: torch.Tensor,
        neg_item_idx: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        user_z, item_z = self.encode_full_graph()
        user_vec = user_z[user_idx]
        pos_scores = (user_vec * item_z[pos_item_idx]).sum(dim=-1)
        neg_scores = (user_vec * item_z[neg_item_idx]).sum(dim=-1)
        return pos_scores, neg_scores

    def score_all_items(self, user_indices: torch.Tensor) -> torch.Tensor:
        user_z, item_z = self.encode_full_graph()
        return user_z[user_indices] @ item_z.T

In [ ]:
def make_sparse_coo(indices: np.ndarray, values: np.ndarray, size: Tuple[int, int], device: torch.device) -> torch.Tensor:
    idx = torch.tensor(indices, dtype=torch.long, device=device)
    vals = torch.tensor(values, dtype=torch.float32, device=device)
    return torch.sparse_coo_tensor(idx, vals, size=size, device=device).coalesce()


def build_relation_graph(
    train_df: pd.DataFrame,
    num_users: int,
    num_items: int,
    rating_values: List[float],
    device: torch.device,
    apply_time_decay: bool = False,
    time_weight_fn_name: str = "exp",
    gamma: float = 1e-8,
) -> dict:
    latest_ts = int(train_df["timestamp"].max()) if len(train_df) else 0
    adjs_u_from_i: List[torch.Tensor] = []
    adjs_i_from_u: List[torch.Tensor] = []
    for rating in rating_values:
        rel_df = train_df[train_df["rating"] == rating]
        if len(rel_df) == 0:
            empty_ui = make_sparse_coo(np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32), (num_users, num_items), device)
            empty_iu = make_sparse_coo(np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32), (num_items, num_users), device)
            adjs_u_from_i.append(empty_ui)
            adjs_i_from_u.append(empty_iu)
            continue
        u = rel_df["user_idx"].astype(np.int64).to_numpy()
        i = rel_df["item_idx"].astype(np.int64).to_numpy()
        w = np.ones(len(rel_df), dtype=np.float32)
        if apply_time_decay:
            delta = (latest_ts - rel_df["timestamp"].astype(np.int64).to_numpy()).clip(min=0)
            w *= time_weight(delta, time_weight_fn_name, gamma)

        user_deg = np.bincount(u, weights=w, minlength=num_users).astype(np.float32)
        item_deg = np.bincount(i, weights=w, minlength=num_items).astype(np.float32)
        norm = w / np.sqrt(np.maximum(user_deg[u], 1e-8) * np.maximum(item_deg[i], 1e-8))

        ui_idx = np.vstack([u, i])
        iu_idx = np.vstack([i, u])
        a_ui = make_sparse_coo(ui_idx, norm, (num_users, num_items), device)
        a_iu = make_sparse_coo(iu_idx, norm, (num_items, num_users), device)
        adjs_u_from_i.append(a_ui)
        adjs_i_from_u.append(a_iu)
    return {"adjs_u_from_i": adjs_u_from_i, "adjs_i_from_u": adjs_i_from_u}


class CometLogger:
    def __init__(self, cfg: ExperimentConfig):
        self.cfg = cfg
        self.enabled = bool(cfg.comet.enabled and COMET_AVAILABLE and os.getenv("COMET_API_KEY"))
        self.exp = None
        if self.enabled:
            self.exp = Experiment(
                api_key=os.getenv("COMET_API_KEY"),
                project_name=cfg.comet.project_name,
                workspace=cfg.comet.workspace,
                log_code=cfg.comet.log_code,
            )
            if cfg.comet.experiment_name:
                self.exp.set_name(cfg.comet.experiment_name)
            if cfg.comet.tags:
                self.exp.add_tags(cfg.comet.tags)
            self.exp.log_parameters(dc_to_dict(cfg))
        else:
            reason = []
            if not cfg.comet.enabled:
                reason.append("disabled in config")
            if not COMET_AVAILABLE:
                reason.append("comet_ml import failed")
            if not os.getenv("COMET_API_KEY"):
                reason.append("COMET_API_KEY is not set")
            print(f"[comet] offline mode: {', '.join(reason)}")

    def log_metrics(self, metrics: dict, step: Optional[int] = None, epoch: Optional[int] = None, prefix: str = "") -> None:
        if prefix:
            metrics = {f"{prefix}{k}": v for k, v in metrics.items()}
        if self.exp is not None:
            self.exp.log_metrics(metrics, step=step, epoch=epoch)

    def log_text(self, name: str, text: str) -> None:
        if self.exp is not None:
            self.exp.log_text(text, metadata={"name": name})

    def end(self):
        if self.exp is not None:
            self.exp.end()

In [ ]:
def bpr_loss(pos_scores: torch.Tensor, neg_scores: torch.Tensor, sample_weights: Optional[torch.Tensor] = None) -> torch.Tensor:
    loss = -F.logsigmoid(pos_scores - neg_scores)
    if sample_weights is not None:
        loss = loss * sample_weights
    return loss.mean()


def to_device(batch: dict, device: torch.device) -> dict:
    out = {}
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.to(device, non_blocking=(device.type == "cuda"))
        else:
            out[k] = v
    return out


def compute_support_alignment_loss(
    model: SupportAwareGCMCModel,
    data: PreparedData,
    cfg: ExperimentConfig,
    device: torch.device,
    user_z: torch.Tensor,
    item_z: torch.Tensor,
) -> torch.Tensor:
    if cfg.model.graph.support_alignment_weight <= 0:
        return user_z.new_tensor(0.0)
    train_users = sorted(data.user_histories.keys())
    if not train_users:
        return user_z.new_tensor(0.0)
    sample_size = min(len(train_users), int(cfg.model.graph.support_user_batch_size))
    if sample_size <= 0:
        return user_z.new_tensor(0.0)
    if sample_size < len(train_users):
        sampled_users = random.sample(train_users, sample_size)
    else:
        sampled_users = train_users
    hist_batch = build_eval_history_tensors(sampled_users, data.user_histories, cfg.train.max_history, device)
    support_user_z = model.encode_support_users(**hist_batch, item_vectors=item_z)
    target_user_z = user_z[torch.tensor(sampled_users, dtype=torch.long, device=device)]
    loss_kind = cfg.model.graph.support_alignment_loss
    if loss_kind == "cosine":
        return (1.0 - F.cosine_similarity(support_user_z, target_user_z, dim=-1)).mean()
    return F.mse_loss(support_user_z, target_user_z)


def train_one_epoch_pairwise(model, loader, optimizer, cfg: ExperimentConfig, logger: CometLogger, epoch: int) -> Dict[str, float]:
    device = torch.device(cfg.train.device)
    model.train()
    scaler = make_grad_scaler(cfg, device)
    total_loss = 0.0
    total_steps = 0
    total_examples = 0
    step_times = []
    grad_norms = []
    pbar = tqdm(loader, desc=f"Train epoch {epoch}", disable=not cfg.train.verbose)
    for step, batch in enumerate(pbar, start=1):
        step_start = time.perf_counter()
        batch = to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)

        with get_autocast_context(cfg, device):
            if cfg.model.model_type == "mf":
                pos_scores, neg_scores = model(batch["user_idx"], batch["pos_item_idx"], batch["neg_item_idx"])
            elif cfg.model.model_type == "two_tower":
                pos_scores, neg_scores = model(
                    batch["user_idx"],
                    batch["pos_item_idx"],
                    batch["neg_item_idx"],
                    batch["hist_items"],
                    batch["hist_ratings"],
                    batch["hist_timestamps"],
                    batch["hist_mask"],
                    batch["target_timestamp"],
                )
            else:
                raise ValueError(f"train_one_epoch_pairwise is not for model_type={cfg.model.model_type}")

            sample_w = rating_weight(batch["rating"], cfg.train.train_rating_weighting)
            loss = bpr_loss(pos_scores, neg_scores, sample_w)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = compute_grad_norm(model.parameters())
        if cfg.train.grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.train.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(loss.item())
        total_steps += 1
        total_examples += int(batch["user_idx"].shape[0])
        step_time = time.perf_counter() - step_start
        step_times.append(step_time)
        grad_norms.append(grad_norm)
        pbar.set_postfix(loss=f"{loss.item():.4f}", grad=f"{grad_norm:.3f}")

        if step % cfg.train.log_every_n_steps == 0:
            logger.log_metrics(
                {
                    "train/loss_step": float(loss.item()),
                    "train/step_time_sec": float(step_time),
                    "train/grad_norm_step": float(grad_norm),
                    "train/examples_per_sec": float(batch["user_idx"].shape[0] / max(step_time, 1e-8)),
                },
                step=(epoch - 1) * len(loader) + step,
                epoch=epoch,
            )

    return {
        "train_loss": float(total_loss / max(total_steps, 1)),
        "train_step_time_mean_sec": float(np.mean(step_times)) if step_times else 0.0,
        "train_grad_norm_mean": float(np.mean(grad_norms)) if grad_norms else 0.0,
        "train_grad_norm_max": float(np.max(grad_norms)) if grad_norms else 0.0,
        "train_examples_per_sec": float(total_examples / max(np.sum(step_times), 1e-8)) if step_times else 0.0,
        "train_steps": int(total_steps),
    }


def train_one_epoch_relation_gcmc(
    model: RelationGCMCModel,
    edge_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    cfg: ExperimentConfig,
    data: PreparedData,
    logger: CometLogger,
    epoch: int,
) -> Dict[str, float]:
    device = torch.device(cfg.train.device)
    model.train()
    scaler = make_grad_scaler(cfg, device)
    optimizer.zero_grad(set_to_none=True)

    forward_start = time.perf_counter()
    with nullcontext():
        user_z, item_z = model.encode_full_graph()
    forward_time = time.perf_counter() - forward_start

    total_loss = 0.0
    total_edges = 0
    num_batches = 0
    align_loss_value = 0.0
    per_batch_backward = bool(getattr(cfg.train, "relation_backward_per_batch", True))
    total_graph_batches = len(edge_loader)

    backward_start = time.perf_counter()
    pbar = tqdm(edge_loader, desc=f"Graph loss epoch {epoch}", disable=not cfg.train.verbose)
    for step, batch in enumerate(pbar, start=1):
        batch = to_device(batch, device)
        with nullcontext():
            logits = model.decode(user_z[batch["user_idx"]], item_z[batch["item_idx"]])
            ce = F.cross_entropy(logits, batch["rating_class"], reduction="mean")
            if cfg.model.graph.aux_bpr_weight > 0:
                raise NotImplementedError("Auxiliary BPR for relation_gcmc is left for a future extension.")
            loss = ce

        batch_weight = batch["user_idx"].shape[0] / len(edge_loader.dataset)
        weighted_loss = loss * batch_weight
        total_loss += float(loss.item()) * batch["user_idx"].shape[0]
        total_edges += int(batch["user_idx"].shape[0])
        num_batches += 1

        if per_batch_backward:
            need_align = hasattr(unwrap_model(model), "encode_support_users") and cfg.model.graph.support_alignment_weight > 0
            retain_graph = (step < total_graph_batches) or need_align
            scaler.scale(weighted_loss).backward(retain_graph=retain_graph)

        if step % max(1, cfg.train.graph_loss_batches_per_epoch_log) == 0:
            logger.log_metrics(
                {
                    "train/relation_batch_ce": float(loss.item()),
                },
                step=(epoch - 1) * len(edge_loader) + step,
                epoch=epoch,
            )
        pbar.set_postfix(ce=f"{loss.item():.4f}")

        del logits, ce, loss, weighted_loss, batch

    base_graph_model = unwrap_model(model)
    if hasattr(base_graph_model, "encode_support_users") and cfg.model.graph.support_alignment_weight > 0:
        with nullcontext():
            align_loss = compute_support_alignment_loss(base_graph_model, data, cfg, device, user_z, item_z)
        align_loss_value = float(align_loss.item())
        scaler.scale(cfg.model.graph.support_alignment_weight * align_loss).backward()
        del align_loss

    scaler.unscale_(optimizer)
    grad_norm = compute_grad_norm(model.parameters())
    if cfg.train.grad_clip_norm is not None:
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.train.grad_clip_norm)
    scaler.step(optimizer)
    scaler.update()
    backward_time = time.perf_counter() - backward_start

    del user_z, item_z
    if device.type == "cuda" and getattr(cfg.train, "empty_cache_each_epoch", False):
        torch.cuda.empty_cache()

    return {
        "train_loss": float(total_loss / max(total_edges, 1)),
        "graph_forward_time_sec": float(forward_time),
        "graph_backward_time_sec": float(backward_time),
        "train_grad_norm_mean": float(grad_norm),
        "train_grad_norm_max": float(grad_norm),
        "train_edges_per_sec": float(total_edges / max(forward_time + backward_time, 1e-8)),
        "support_alignment_loss": float(align_loss_value),
        "train_batches": int(num_batches),
    }


def train_one_epoch_ordinal_ngcf(
    model: OrdinalNGCFModel,
    pairwise_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    cfg: ExperimentConfig,
    logger: CometLogger,
    epoch: int,
) -> Dict[str, float]:
    """
    Full-graph NGCF encoder + BPR mini-batches.

    We compute graph embeddings once per epoch and accumulate BPR gradients
    over pairwise batches. This is the same memory pattern as relation_gcmc:
    the graph computation is retained until the final backward call.
    """
    device = torch.device(cfg.train.device)
    model.train()
    scaler = make_grad_scaler(cfg, device)
    optimizer.zero_grad(set_to_none=True)

    forward_start = time.perf_counter()
    with nullcontext():
        user_z, item_z = model.encode_full_graph()
    forward_time = time.perf_counter() - forward_start

    total_loss = 0.0
    total_examples = 0
    total_batches = len(pairwise_loader)
    grad_norm = 0.0

    backward_start = time.perf_counter()
    pbar = tqdm(pairwise_loader, desc=f"Ordinal NGCF loss epoch {epoch}", disable=not cfg.train.verbose)

    for step, batch in enumerate(pbar, start=1):
        batch = to_device(batch, device)

        user_idx = batch["user_idx"]
        pos_item_idx = batch["pos_item_idx"]
        neg_item_idx = batch["neg_item_idx"]

        user_vec = user_z[user_idx]
        pos_scores = (user_vec * item_z[pos_item_idx]).sum(dim=-1)
        neg_scores = (user_vec * item_z[neg_item_idx]).sum(dim=-1)

        sample_w = rating_weight(batch["rating"], cfg.train.train_rating_weighting)
        loss = bpr_loss(pos_scores, neg_scores, sample_w)

        batch_weight = user_idx.shape[0] / len(pairwise_loader.dataset)
        weighted_loss = loss * batch_weight

        retain_graph = step < total_batches
        scaler.scale(weighted_loss).backward(retain_graph=retain_graph)

        total_loss += float(loss.item()) * int(user_idx.shape[0])
        total_examples += int(user_idx.shape[0])

        if step % max(1, cfg.train.graph_loss_batches_per_epoch_log) == 0:
            logger.log_metrics(
                {
                    "train/ordinal_ngcf_batch_bpr": float(loss.item()),
                },
                step=(epoch - 1) * len(pairwise_loader) + step,
                epoch=epoch,
            )

        pbar.set_postfix(bpr=f"{loss.item():.4f}")

        del batch, user_idx, pos_item_idx, neg_item_idx, user_vec, pos_scores, neg_scores, loss, weighted_loss

    scaler.unscale_(optimizer)
    grad_norm = compute_grad_norm(model.parameters())

    if cfg.train.grad_clip_norm is not None:
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.train.grad_clip_norm)

    scaler.step(optimizer)
    scaler.update()

    backward_time = time.perf_counter() - backward_start

    del user_z, item_z
    if device.type == "cuda" and getattr(cfg.train, "empty_cache_each_epoch", False):
        torch.cuda.empty_cache()

    return {
        "train_loss": float(total_loss / max(total_examples, 1)),
        "graph_forward_time_sec": float(forward_time),
        "graph_backward_time_sec": float(backward_time),
        "train_grad_norm_mean": float(grad_norm),
        "train_grad_norm_max": float(grad_norm),
        "train_examples_per_sec": float(total_examples / max(forward_time + backward_time, 1e-8)),
        "train_examples": int(total_examples),
        "train_batches": int(total_batches),
    }


def get_rating_weight_table_metrics(model: nn.Module, prefix: str = "rating_weight") -> Dict[str, float]:
    """
    Returns rating -> learned edge weight table for Ordinal NGCF.
    Logged to Comet each epoch. For linear_norm this logs the fixed baseline curve;
    for exp_mixture / softplus_spline this logs the learned curve.
    """
    base_model = unwrap_model(model)
    if not hasattr(base_model, "edge_weight_fn"):
        return {}

    edge_weight_fn = base_model.edge_weight_fn
    if not hasattr(edge_weight_fn, "current_table"):
        return {}

    try:
        table = edge_weight_fn.current_table()
    except Exception as exc:
        print(f"[warn] could not log rating weight table: {exc}")
        return {}

    weight_metrics = {f"{prefix}/{r:.1f}": float(w) for r, w in table.items()}

    # A couple of shape diagnostics for learned smooth functions.
    if hasattr(edge_weight_fn, "fn") and isinstance(edge_weight_fn.fn, ExpMixtureRatingWeight):
        fn = edge_weight_fn.fn
        lambdas = torch.clamp(fn.raw_lambdas.detach(), min=-fn.lambda_clip, max=fn.lambda_clip).cpu().numpy()
        pis = torch.softmax(fn.logits.detach(), dim=0).cpu().numpy()
        for idx, value in enumerate(lambdas):
            weight_metrics[f"{prefix}_exp/lambda_{idx}"] = float(value)
        for idx, value in enumerate(pis):
            weight_metrics[f"{prefix}_exp/pi_{idx}"] = float(value)

    if hasattr(edge_weight_fn, "fn") and isinstance(edge_weight_fn.fn, SoftplusSplineRatingWeight):
        fn = edge_weight_fn.fn
        try:
            kappa = float(fn._get_kappa().detach().cpu().item())
            weight_metrics[f"{prefix}_spline/kappa"] = kappa
        except Exception:
            pass

    return weight_metrics


def build_eval_history_tensors(
    user_indices: Sequence[int],
    user_histories: Dict[int, List[Tuple[int, float, int]]],
    max_history: int,
    device: torch.device,
) -> dict:
    batch_hist = [user_histories.get(int(u), [])[-max_history:] for u in user_indices]
    max_len = max((len(h) for h in batch_hist), default=1)
    hist_items = torch.zeros((len(user_indices), max_len), dtype=torch.long, device=device)
    hist_ratings = torch.zeros((len(user_indices), max_len), dtype=torch.float32, device=device)
    hist_timestamps = torch.zeros((len(user_indices), max_len), dtype=torch.long, device=device)
    hist_mask = torch.zeros((len(user_indices), max_len), dtype=torch.bool, device=device)
    target_timestamp = torch.zeros((len(user_indices),), dtype=torch.long, device=device)

    for row, hist in enumerate(batch_hist):
        if not hist:
            continue
        items, ratings, timestamps = zip(*hist)
        n = len(items)
        hist_items[row, :n] = torch.tensor(items, dtype=torch.long, device=device)
        hist_ratings[row, :n] = torch.tensor(ratings, dtype=torch.float32, device=device)
        hist_timestamps[row, :n] = torch.tensor(timestamps, dtype=torch.long, device=device)
        hist_mask[row, :n] = True
        target_timestamp[row] = int(max(timestamps))
    return {
        "hist_items": hist_items,
        "hist_ratings": hist_ratings,
        "hist_timestamps": hist_timestamps,
        "hist_mask": hist_mask,
        "target_timestamp": target_timestamp,
    }


def build_eval_history_tensors_for_rows(
    user_indices: Sequence[int],
    target_timestamps: Sequence[int],
    user_histories: Dict[int, List[Tuple[int, float, int]]],
    max_history: int,
    device: torch.device,
) -> dict:
    batch_hist = [user_histories.get(int(u), [])[-max_history:] for u in user_indices]
    max_len = max((len(h) for h in batch_hist), default=1)
    hist_items = torch.zeros((len(user_indices), max_len), dtype=torch.long, device=device)
    hist_ratings = torch.zeros((len(user_indices), max_len), dtype=torch.float32, device=device)
    hist_timestamps = torch.zeros((len(user_indices), max_len), dtype=torch.long, device=device)
    hist_mask = torch.zeros((len(user_indices), max_len), dtype=torch.bool, device=device)
    target_timestamp = torch.tensor(list(target_timestamps), dtype=torch.long, device=device)

    for row, hist in enumerate(batch_hist):
        if not hist:
            continue
        items, ratings, timestamps = zip(*hist)
        n = len(items)
        hist_items[row, :n] = torch.tensor(items, dtype=torch.long, device=device)
        hist_ratings[row, :n] = torch.tensor(ratings, dtype=torch.float32, device=device)
        hist_timestamps[row, :n] = torch.tensor(timestamps, dtype=torch.long, device=device)
        hist_mask[row, :n] = True
    return {
        "hist_items": hist_items,
        "hist_ratings": hist_ratings,
        "hist_timestamps": hist_timestamps,
        "hist_mask": hist_mask,
        "target_timestamp": target_timestamp,
    }


def _stable_seed_from_text(text: str, base_seed: int = 42) -> int:
    seed = int(base_seed) & 0xFFFFFFFF
    for ch in text:
        seed = (seed * 131 + ord(ch)) & 0xFFFFFFFF
    return seed


def _sample_negative_item(user_seen: set, num_items: int, rng: np.random.Generator) -> int:
    if len(user_seen) >= num_items:
        return int(rng.integers(0, num_items))
    neg = int(rng.integers(0, num_items))
    max_tries = 64
    tries = 0
    while neg in user_seen and tries < max_tries:
        neg = int(rng.integers(0, num_items))
        tries += 1
    if neg in user_seen:
        candidates = np.setdiff1d(np.arange(num_items, dtype=np.int64), np.fromiter(user_seen, dtype=np.int64), assume_unique=False)
        if len(candidates) == 0:
            return int(rng.integers(0, num_items))
        neg = int(candidates[int(rng.integers(0, len(candidates)))])
    return neg


def compute_eval_loss(model, data: PreparedData, cfg: ExperimentConfig, eval_key: str) -> Dict[str, float]:
    if not cfg.eval.compute_loss:
        return {}
    if eval_key not in data.eval_frames or not model_supports_eval_key(cfg.model.model_type, eval_key):
        return {}

    eval_df = data.eval_frames[eval_key]
    if eval_df.empty:
        return {}

    if cfg.eval.loss_max_rows is not None and len(eval_df) > cfg.eval.loss_max_rows:
        eval_df = eval_df.sample(cfg.eval.loss_max_rows, random_state=cfg.train.seed).reset_index(drop=True)

    device = torch.device(cfg.train.device)
    base_model = unwrap_model(model)
    base_model.eval()
    user_histories = data.eval_histories.get(eval_key, data.user_histories)
    base_seen_items = data.eval_seen_items.get(eval_key, data.seen_items)
    observed_eval_items = defaultdict(set)
    for row in eval_df[["user_idx", "item_idx"]].itertuples(index=False):
        observed_eval_items[int(row.user_idx)].add(int(row.item_idx))
    exclusion_by_user = {
        int(u): set(base_seen_items.get(int(u), set())) | observed_eval_items.get(int(u), set())
        for u in set(base_seen_items.keys()) | set(observed_eval_items.keys())
    }

    if cfg.model.model_type in {"mf", "two_tower"}:
        pos_df = eval_df.loc[eval_df["rating"] >= cfg.train.positive_threshold, ["user_idx", "item_idx", "rating", "timestamp"]].reset_index(drop=True)
        if pos_df.empty:
            return {"loss": 0.0, "loss_rows": 0}
        rng = np.random.default_rng(_stable_seed_from_text(eval_key, cfg.train.seed))
        total_loss = 0.0
        total_rows = 0
        batch_size = max(1, cfg.eval.loss_batch_size)
        with torch.no_grad():
            for start in range(0, len(pos_df), batch_size):
                batch_df = pos_df.iloc[start: start + batch_size]
                user_idx = batch_df["user_idx"].to_numpy(dtype=np.int64)
                pos_item_idx = batch_df["item_idx"].to_numpy(dtype=np.int64)
                ratings = batch_df["rating"].to_numpy(dtype=np.float32)
                timestamps = batch_df["timestamp"].to_numpy(dtype=np.int64)
                neg_item_idx = np.array([
                    _sample_negative_item(exclusion_by_user.get(int(u), set()), data.num_items, rng)
                    for u in user_idx
                ], dtype=np.int64)

                user_t = torch.tensor(user_idx, dtype=torch.long, device=device)
                pos_t = torch.tensor(pos_item_idx, dtype=torch.long, device=device)
                neg_t = torch.tensor(neg_item_idx, dtype=torch.long, device=device)
                rating_t = torch.tensor(ratings, dtype=torch.float32, device=device)

                with get_autocast_context(cfg, device):
                    if cfg.model.model_type == "mf":
                        pos_scores = base_model.score(user_t, pos_t)
                        neg_scores = base_model.score(user_t, neg_t)
                    else:
                        hist_batch = build_eval_history_tensors_for_rows(
                            user_indices=user_idx.tolist(),
                            target_timestamps=timestamps.tolist(),
                            user_histories=user_histories,
                            max_history=cfg.train.max_history,
                            device=device,
                        )
                        user_vec = base_model.encode_user(user_t, **hist_batch)
                        pos_scores = (user_vec * base_model.encode_item(pos_t)).sum(dim=-1)
                        neg_scores = (user_vec * base_model.encode_item(neg_t)).sum(dim=-1)
                    sample_w = rating_weight(rating_t, cfg.train.train_rating_weighting)
                    loss = bpr_loss(pos_scores, neg_scores, sample_w)
                batch_rows = len(batch_df)
                total_loss += float(loss.item()) * batch_rows
                total_rows += batch_rows
        return {"loss": float(total_loss / max(total_rows, 1)), "loss_rows": int(total_rows)}


    if cfg.model.model_type == "ordinal_ngcf":
        pos_df = eval_df.loc[eval_df["rating"] >= cfg.train.positive_threshold, ["user_idx", "item_idx", "rating", "timestamp"]].reset_index(drop=True)
        if pos_df.empty:
            return {"loss": 0.0, "loss_rows": 0}

        rng = np.random.default_rng(_stable_seed_from_text(eval_key, cfg.train.seed))
        total_loss = 0.0
        total_rows = 0
        batch_size = max(1, cfg.eval.loss_batch_size)

        with torch.no_grad():
            with nullcontext():
                graph_user_z, graph_item_z = base_model.encode_full_graph()

            for start in range(0, len(pos_df), batch_size):
                batch_df = pos_df.iloc[start: start + batch_size]
                user_idx = batch_df["user_idx"].to_numpy(dtype=np.int64)
                pos_item_idx = batch_df["item_idx"].to_numpy(dtype=np.int64)
                ratings = batch_df["rating"].to_numpy(dtype=np.float32)

                neg_item_idx = np.array([
                    _sample_negative_item(exclusion_by_user.get(int(u), set()), data.num_items, rng)
                    for u in user_idx
                ], dtype=np.int64)

                user_t = torch.tensor(user_idx, dtype=torch.long, device=device)
                pos_t = torch.tensor(pos_item_idx, dtype=torch.long, device=device)
                neg_t = torch.tensor(neg_item_idx, dtype=torch.long, device=device)
                rating_t = torch.tensor(ratings, dtype=torch.float32, device=device)

                user_vec = graph_user_z[user_t]
                pos_scores = (user_vec * graph_item_z[pos_t]).sum(dim=-1)
                neg_scores = (user_vec * graph_item_z[neg_t]).sum(dim=-1)

                sample_w = rating_weight(rating_t, cfg.train.train_rating_weighting)
                loss = bpr_loss(pos_scores, neg_scores, sample_w)

                batch_rows = len(batch_df)
                total_loss += float(loss.item()) * batch_rows
                total_rows += batch_rows

        return {"loss": float(total_loss / max(total_rows, 1)), "loss_rows": int(total_rows)}

    if cfg.model.model_type in {"relation_gcmc", "support_gcmc"}:
        valid_mask = eval_df["rating"].isin(data.rating2class.keys())
        eval_obs = eval_df.loc[valid_mask, ["user_idx", "item_idx", "rating", "timestamp"]].reset_index(drop=True)
        if eval_obs.empty:
            return {"loss": 0.0, "loss_rows": 0}
        graph_batch_size = max(1, cfg.eval.loss_batch_size)
        total_loss = 0.0
        total_rows = 0
        with torch.no_grad():
            with nullcontext():
                graph_user_z, graph_item_z = base_model.encode_full_graph()
            for start in range(0, len(eval_obs), graph_batch_size):
                batch_df = eval_obs.iloc[start: start + graph_batch_size]
                user_idx = batch_df["user_idx"].to_numpy(dtype=np.int64)
                item_idx = batch_df["item_idx"].to_numpy(dtype=np.int64)
                timestamps = batch_df["timestamp"].to_numpy(dtype=np.int64)
                rating_class = np.array([data.rating2class[float(r)] for r in batch_df["rating"].tolist()], dtype=np.int64)

                user_t = torch.tensor(user_idx, dtype=torch.long, device=device)
                item_t = torch.tensor(item_idx, dtype=torch.long, device=device)
                class_t = torch.tensor(rating_class, dtype=torch.long, device=device)

                with get_autocast_context(cfg, device):
                    if cfg.model.model_type == "support_gcmc" and (eval_key.startswith("cold_user_") or eval_key.startswith("both_cold_")):
                        hist_batch = build_eval_history_tensors_for_rows(
                            user_indices=user_idx.tolist(),
                            target_timestamps=timestamps.tolist(),
                            user_histories=user_histories,
                            max_history=cfg.train.max_history,
                            device=device,
                        )
                        user_vec = base_model.encode_support_users(**hist_batch, item_vectors=graph_item_z)
                    else:
                        user_vec = graph_user_z[user_t]
                    logits = base_model.decode(user_vec, graph_item_z[item_t])
                    loss = F.cross_entropy(logits, class_t, reduction="mean")
                batch_rows = len(batch_df)
                total_loss += float(loss.item()) * batch_rows
                total_rows += batch_rows
        return {"loss": float(total_loss / max(total_rows, 1)), "loss_rows": int(total_rows)}

    return {}


def get_eval_metric_names(cfg_eval: EvalConfig, gain_type: str) -> Tuple[str, ...]:
    family = gain_type_metric_family(gain_type)
    if family == "binary":
        return tuple(cfg_eval.binary_metric_names)
    return tuple(cfg_eval.graded_metric_names)


def get_eval_keys(cfg: ExperimentConfig, split_name: str) -> List[str]:
    keys = [f"warm_{split_name}"]
    include_cold = cfg.eval.evaluate_cold_on_val if split_name == "val" else cfg.eval.evaluate_cold_on_test
    if not include_cold:
        return keys
    for scenario in cfg.eval.eval_scenarios:
        if scenario == "warm":
            continue
        keys.append(f"{scenario}_{split_name}")
    return keys


def model_supports_eval_key(model_type: str, eval_key: str) -> bool:
    if eval_key.startswith("warm_"):
        return True
    if eval_key.startswith("cold_item_"):
        return model_type in {"mf", "two_tower", "relation_gcmc", "support_gcmc", "ordinal_ngcf"}
    if eval_key.startswith("cold_user_") or eval_key.startswith("both_cold_"):
        return model_type in {"two_tower", "support_gcmc", "ordinal_ngcf"}
    return False


import numpy as np
from typing import Dict, Sequence


def dcg_at_k(gains: Sequence[float], k: int) -> float:
    gains = np.asarray(gains[:k], dtype=np.float32)
    if gains.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, gains.size + 2))
    return float(np.sum(gains * discounts))


def compute_ranking_metrics(
    ranked_items: np.ndarray,
    target_gain_dict: Dict[int, float],
    ks: Sequence[int],
    metric_names: Sequence[str],
) -> Dict[str, float]:
    metrics: Dict[str, float] = {}

    gains_ranked = np.asarray(
        [target_gain_dict.get(int(i), 0.0) for i in ranked_items],
        dtype=np.float32,
    )
    binary_ranked = (gains_ranked > 0).astype(np.float32)
    relevant_total = int(
        np.sum(np.asarray(list(target_gain_dict.values()), dtype=np.float32) > 0)
    )
    ideal_gains = sorted(target_gain_dict.values(), reverse=True)

    for k in ks:
        topk_gains = gains_ranked[:k]
        topk_binary = binary_ranked[:k]
        hits = int(np.sum(topk_binary))

        if "hitrate" in metric_names:
            metrics[f"hitrate@{k}"] = float(hits > 0)

        if "precision" in metric_names:
            metrics[f"precision@{k}"] = float(hits / max(k, 1))

        if "recall" in metric_names:
            metrics[f"recall@{k}"] = float(hits / relevant_total) if relevant_total > 0 else 0.0

        if "ndcg" in metric_names:
            dcg = dcg_at_k(topk_gains, k)
            idcg = dcg_at_k(ideal_gains, min(k, len(ideal_gains)))
            metrics[f"ndcg@{k}"] = float(dcg / idcg) if idcg > 0 else 0.0

        if "mrr" in metric_names or "map" in metric_names:
            rel_positions = np.flatnonzero(topk_binary > 0)

            if "mrr" in metric_names:
                metrics[f"mrr@{k}"] = float(1.0 / (rel_positions[0] + 1)) if rel_positions.size > 0 else 0.0

            if "map" in metric_names:
                if rel_positions.size > 0:
                    prefix_hits = np.cumsum(topk_binary)
                    precisions = prefix_hits[rel_positions] / (rel_positions + 1)
                    ap_denom = min(relevant_total, k)
                    metrics[f"map@{k}"] = float(np.sum(precisions) / ap_denom) if ap_denom > 0 else 0.0
                else:
                    metrics[f"map@{k}"] = 0.0

    return metrics


def evaluate_ranked_items(
    ranked_items: np.ndarray,
    target_gain_by_item: Dict[int, float],
    ks: Sequence[int],
    metric_names: Sequence[str],
) -> Dict[str, float]:
    return compute_ranking_metrics(
        ranked_items=ranked_items,
        target_gain_dict=target_gain_by_item,
        ks=ks,
        metric_names=metric_names,
    )





@torch.inference_mode()
def evaluate_model(model, data: PreparedData, cfg: ExperimentConfig, eval_key: str = "warm_val") -> Dict[str, float]:
    if eval_key not in data.eval_frames:
        return {"skipped": 1.0}
    if not model_supports_eval_key(cfg.model.model_type, eval_key):
        return {"skipped": 1.0}

    base_model = unwrap_model(model)
    device = torch.device(cfg.train.device)
    base_model.eval()
    eval_df = data.eval_frames[eval_key]
    if eval_df.empty:
        return {"users_scored": 0, "time_sec": 0.0, "empty": 1.0}

    gain_types = get_metric_gain_types(cfg.eval)
    targets_by_gain = {gain_type: build_eval_targets(eval_df, cfg.eval, gain_type) for gain_type in gain_types}
    users = sorted(set().union(*[set(v.keys()) for v in targets_by_gain.values()]))
    if not users:
        return {}

    metric_rows_by_gain: Dict[str, List[Dict[str, float]]] = {gain_type: [] for gain_type in gain_types}
    user_histories = data.eval_histories.get(eval_key, data.user_histories)
    seen_items_by_user = data.eval_seen_items.get(eval_key, data.seen_items)
    t0 = time.perf_counter()

    graph_user_z = None
    graph_item_z = None
    if cfg.model.model_type in {"relation_gcmc", "support_gcmc", "ordinal_ngcf"}:
        with torch.no_grad():
            with nullcontext():
                graph_user_z, graph_item_z = base_model.encode_full_graph()

    for start_idx in tqdm(range(0, len(users), cfg.eval.user_batch_size), desc=f"Eval {eval_key}"):
        batch_users = users[start_idx: start_idx + cfg.eval.user_batch_size]
        user_tensor = torch.tensor(batch_users, dtype=torch.long, device=device)

        with get_autocast_context(cfg, device):
            if cfg.model.model_type == "mf":
                score_matrix = base_model.score_all_items(user_tensor)
            elif cfg.model.model_type == "two_tower":
                hist_batch = build_eval_history_tensors(batch_users, user_histories, cfg.train.max_history, device)
                user_vecs = base_model.encode_user(user_tensor, **hist_batch)
                score_matrix = base_model.score_all_items_for_histories(user_vecs)
            elif cfg.model.model_type == "relation_gcmc":
                score_matrix = base_model.score_user_vectors_all_items(graph_user_z[user_tensor], graph_item_z, positive_threshold=cfg.eval.positive_threshold, item_batch_size=cfg.eval.item_batch_size)
            elif cfg.model.model_type == "ordinal_ngcf":
                score_matrix = graph_user_z[user_tensor] @ graph_item_z.T
            elif cfg.model.model_type == "support_gcmc":
                if eval_key.startswith("cold_user_") or eval_key.startswith("both_cold_"):
                    hist_batch = build_eval_history_tensors(batch_users, user_histories, cfg.train.max_history, device)
                    user_vecs = base_model.encode_support_users(**hist_batch, item_vectors=graph_item_z)
                else:
                    user_vecs = graph_user_z[user_tensor]
                score_matrix = base_model.score_user_vectors_all_items(user_vecs, graph_item_z, positive_threshold=cfg.eval.positive_threshold, item_batch_size=cfg.eval.item_batch_size)
            else:
                raise ValueError(f"Unsupported model_type={cfg.model.model_type}")

        score_matrix = score_matrix.detach().float().cpu().numpy().astype(np.float32)
        max_k = max(cfg.eval.ks)

        for row_idx, user_idx in enumerate(batch_users):
            scores = score_matrix[row_idx]
            if cfg.eval.exclude_seen_items:
                seen = seen_items_by_user.get(int(user_idx), set())
                if seen:
                    seen_idx = np.fromiter(seen, dtype=np.int64)
                    valid_seen = seen_idx[(seen_idx >= 0) & (seen_idx < len(scores))]
                    if valid_seen.size > 0:
                        scores[valid_seen] = -1e9
            kth = min(max_k, len(scores) - 1)
            topk_idx = np.argpartition(-scores, kth=kth)[:max_k]
            ranked_items = topk_idx[np.argsort(-scores[topk_idx])]

            for gain_type in gain_types:
                target_gain = targets_by_gain[gain_type].get(int(user_idx), {})
                if not target_gain and cfg.eval.skip_users_with_no_relevant_targets:
                    continue
                metric_names = get_eval_metric_names(cfg.eval, gain_type)
                row_metrics = evaluate_ranked_items(
                    ranked_items=ranked_items,
                    target_gain_by_item=target_gain,
                    ks=cfg.eval.ks,
                    metric_names=metric_names,
                )
                metric_rows_by_gain[gain_type].append(row_metrics)

    elapsed = time.perf_counter() - t0
    out = {"users_scored": len(users), "time_sec": float(elapsed)}
    loss_metrics = compute_eval_loss(base_model, data, cfg, eval_key)
    out.update(loss_metrics)
    for gain_type, rows in metric_rows_by_gain.items():
        if not rows:
            continue
        df = pd.DataFrame(rows)
        for col in df.columns:
            out[f"{gain_type}/{col}"] = float(df[col].mean())
    return out

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset


def unwrap_model(model: torch.nn.Module) -> torch.nn.Module:
    return model.module if hasattr(model, "module") else model


def maybe_wrap_for_multi_gpu(model: torch.nn.Module, cfg) -> torch.nn.Module:
    use_multi_gpu = bool(getattr(cfg.train, "use_multi_gpu", False))
    if not use_multi_gpu:
        return model
    if not torch.cuda.is_available():
        return model
    if torch.cuda.device_count() < 2:
        return model

    # Реально оборачиваем только те модели, для которых в ноутбуке
    # предполагался multi-GPU режим.
    if getattr(cfg.model, "model_type", None) in {"mf", "two_tower"}:
        return torch.nn.DataParallel(model)

    return model


def maybe_compile_model(model: torch.nn.Module, cfg) -> torch.nn.Module:
    use_compile = bool(
        getattr(cfg.train, "use_torch_compile", False)
        or getattr(cfg.train, "compile_model", False)
    )
    if not use_compile:
        return model

    if not hasattr(torch, "compile"):
        return model

    try:
        return torch.compile(model)
    except Exception as e:
        print(f"[warn] torch.compile disabled: {e}")
        return model


def make_loader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool,
    cfg,
    collate_fn=None,
) -> DataLoader:
    num_workers = max(int(getattr(cfg.train, "num_workers", 0)), 0)

    loader_kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=bool(getattr(cfg.train, "pin_memory", torch.cuda.is_available())),
        collate_fn=collate_fn,
    )

    if num_workers > 0:
        loader_kwargs["persistent_workers"] = bool(
            getattr(cfg.train, "persistent_workers", False)
        )
        prefetch_factor = getattr(cfg.train, "prefetch_factor", None)
        if prefetch_factor is not None:
            loader_kwargs["prefetch_factor"] = int(prefetch_factor)

    return DataLoader(**loader_kwargs)

In [ ]:
def build_model_and_loaders(cfg: ExperimentConfig, data: PreparedData):
    device = torch.device(cfg.train.device)
    warm_candidate_items = np.sort(data.train_df["item_idx"].astype(np.int64).unique())
    pairwise_ds = PairwiseTrainDataset(data.train_df, data.seen_items, warm_candidate_items, cfg.train.positive_threshold)
    pairwise_loader = make_loader(pairwise_ds, cfg.train.batch_size, True, cfg)

    if cfg.model.model_type == "mf":
        model = MFModel(
            data.num_users,
            data.num_items,
            data.item_features,
            cfg.features,
            cfg.model.mf,
            trainable_item_mask=data.trainable_item_mask,
        ).to(device)
        model = maybe_wrap_for_multi_gpu(model, cfg)
        model = maybe_compile_model(model, cfg)
        optimizer = torch.optim.Adam(unwrap_model(model).parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, pairwise_loader

    if cfg.model.model_type == "two_tower":
        seq_ds = SequencePairwiseDataset(
            data.train_df,
            warm_candidate_items,
            cfg.train.positive_threshold,
            cfg.train.max_history,
            cfg.train.min_history_for_sequence,
        )
        seq_loader = make_loader(seq_ds, cfg.train.batch_size, True, cfg, collate_fn=sequence_collate_fn)
        model = TwoTowerModel(
            data.num_users,
            data.num_items,
            data.item_features,
            cfg.features,
            cfg.model.two_tower,
            trainable_item_mask=data.trainable_item_mask,
        ).to(device)
        model = maybe_wrap_for_multi_gpu(model, cfg)
        model = maybe_compile_model(model, cfg)
        optimizer = torch.optim.Adam(unwrap_model(model).parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, seq_loader

    if cfg.model.model_type == "ordinal_ngcf":
        model = OrdinalNGCFModel(
            data.num_users,
            data.num_items,
            data.item_features,
            cfg.features,
            cfg.model.graph,
            data.unique_ratings,
            trainable_item_mask=data.trainable_item_mask,
            num_warm_users=data.num_warm_users,
        ).to(device)

        model.set_graph_edges(data.train_df)
        model.initialize_fixed_cold_user_vector(
            data.train_df,
            positive_threshold=cfg.train.positive_threshold,
        )

        # Sparse full-graph path is single-GPU.
        model = maybe_compile_model(model, cfg)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, pairwise_loader

    if cfg.model.model_type in {"relation_gcmc", "support_gcmc"}:
        edge_ds = ObservedEdgeDataset(data.train_df, data.rating2class)
        edge_loader = make_loader(edge_ds, cfg.train.relation_edge_batch_size, True, cfg)
        model_cls = SupportAwareGCMCModel if cfg.model.model_type == "support_gcmc" else RelationGCMCModel
        model = model_cls(
            data.num_users,
            data.num_items,
            data.item_features,
            cfg.features,
            cfg.model.graph,
            data.unique_ratings,
            trainable_item_mask=data.trainable_item_mask,
        ).to(device)
        graph_dict = build_relation_graph(
            data.train_df,
            data.num_users,
            data.num_items,
            data.unique_ratings,
            device,
            apply_time_decay=cfg.model.graph.relation_apply_time_decay,
            time_weight_fn_name=cfg.model.graph.time_weight_fn,
            gamma=cfg.model.graph.time_weight_gamma,
        )
        model.set_graph(graph_dict)
        model = maybe_compile_model(model, cfg)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        return model, optimizer, edge_loader

    raise ValueError(f"Unknown model_type={cfg.model.model_type}")

In [ ]:
def select_main_metric(metrics: Dict[str, float], cfg_eval: EvalConfig) -> float:
    metric_k = cfg_eval.main_metric_k if cfg_eval.main_metric_k in cfg_eval.ks else cfg_eval.ks[0]
    key = f"{cfg_eval.main_metric_gain_type}/{cfg_eval.main_metric_name}@{metric_k}"
    return float(metrics.get(key, -1e9))


def save_checkpoint(model: nn.Module, output_dir: Path, name: str) -> None:
    path = output_dir / name
    torch.save(unwrap_model(model).state_dict(), path)
    print(f"[checkpoint] saved to {path}")


def run_training(cfg: ExperimentConfig):
    set_seed(cfg.train.seed)
    configure_runtime(cfg)
    data = load_prepared_data(cfg)
    model, optimizer, train_loader = build_model_and_loaders(cfg, data)
    logger = CometLogger(cfg)
    runtime_info = get_gpu_info()
    logger.log_metrics({
        "runtime/num_gpus": runtime_info["num_gpus"],
        "runtime/use_amp": float(cfg.train.use_amp),
        "runtime/use_multi_gpu": float(cfg.train.use_multi_gpu),
    })
    if runtime_info["gpu_names"]:
        logger.log_text("gpu_names", ", ".join(runtime_info["gpu_names"]))
    save_json(dc_to_dict(cfg), data.output_dir / f"config_{cfg.model.model_type}.json")

    best_metric = -1e18
    best_epoch = -1
    best_metrics = None
    patience_counter = 0
    history = []

    val_eval_keys = get_eval_keys(cfg, "val")
    test_eval_keys = get_eval_keys(cfg, "test")

    for epoch in range(1, cfg.train.epochs + 1):
        print("=" * 80)
        print(f"Epoch {epoch}/{cfg.train.epochs} | model_type={cfg.model.model_type}")
        epoch_start = time.perf_counter()

        if cfg.model.model_type in {"mf", "two_tower"}:
            train_metrics = train_one_epoch_pairwise(model, train_loader, optimizer, cfg, logger, epoch)
        elif cfg.model.model_type == "ordinal_ngcf":
            train_metrics = train_one_epoch_ordinal_ngcf(model, train_loader, optimizer, cfg, logger, epoch)
        elif cfg.model.model_type in {"relation_gcmc", "support_gcmc"}:
            train_metrics = train_one_epoch_relation_gcmc(model, train_loader, optimizer, cfg, data, logger, epoch)
        else:
            raise ValueError(cfg.model.model_type)

        val_by_scenario: Dict[str, Dict[str, float]] = {}
        flat_val_metrics: Dict[str, float] = {}
        for eval_key in val_eval_keys:
            scenario_metrics = evaluate_model(model, data, cfg, eval_key=eval_key)
            val_by_scenario[eval_key] = scenario_metrics
            flat_val_metrics.update({f"val/{eval_key}/{k}": v for k, v in scenario_metrics.items()})

        epoch_time = time.perf_counter() - epoch_start
        weight_metrics = get_rating_weight_table_metrics(model)
        epoch_metrics = {**train_metrics, **flat_val_metrics, **weight_metrics, "epoch_time_sec": float(epoch_time)}
        logger.log_metrics(epoch_metrics, epoch=epoch)
        history.append({"epoch": epoch, **epoch_metrics})

        print("TRAIN | " + " | ".join(f"{k}={v:.5f}" if isinstance(v, float) else f"{k}={v}" for k, v in train_metrics.items()))
        for eval_key, scenario_metrics in val_by_scenario.items():
            printable = " | ".join(f"{k}={v:.5f}" if isinstance(v, float) else f"{k}={v}" for k, v in scenario_metrics.items())
            print(f"VAL[{eval_key}] | {printable}")
        print(f"TIME  | epoch_time_sec={epoch_time:.3f}")

        current_metric = select_main_metric(val_by_scenario.get("warm_val", {}), cfg.eval)
        if current_metric > best_metric:
            best_metric = current_metric
            best_epoch = epoch
            best_metrics = val_by_scenario.get("warm_val", {})
            patience_counter = 0
            save_checkpoint(model, data.output_dir, f"best_{cfg.model.model_type}.pt")
        else:
            patience_counter += 1
            print(f"[early-stop] no improvement for {patience_counter} epoch(s)")
            if patience_counter >= cfg.train.patience:
                print("[early-stop] stopping training")
                break

    print("=" * 80)
    print(f"Best epoch={best_epoch} | main_metric={best_metric:.6f}")
    hist_df = pd.DataFrame(history)
    hist_path = data.output_dir / f"history_{cfg.model.model_type}.csv"
    hist_df.to_csv(hist_path, index=False)
    print(f"Saved training history to: {hist_path}")

    best_ckpt = data.output_dir / f"best_{cfg.model.model_type}.pt"
    if best_ckpt.exists():
        unwrap_model(model).load_state_dict(torch.load(best_ckpt, map_location=cfg.train.device))

    test_by_scenario: Dict[str, Dict[str, float]] = {}
    flat_test_metrics: Dict[str, float] = {}
    for eval_key in test_eval_keys:
        scenario_metrics = evaluate_model(model, data, cfg, eval_key=eval_key)
        test_by_scenario[eval_key] = scenario_metrics
        flat_test_metrics.update({f"test/{eval_key}/{k}": v for k, v in scenario_metrics.items()})
    logger.log_metrics(flat_test_metrics)
    logger.end()

    return model, data, hist_df, best_metrics, test_by_scenario

In [ ]:
import os

# Для логирования в Comet задайте ключ вне ноутбука, например:
# export COMET_API_KEY="..."
# На GitHub ключи и токены не хранятся.
os.environ.setdefault("COMET_PROJECT_NAME", "explicit-movie-ranking")


In [ ]:
def make_ordinal_ngcf_cfg(
    weight_fn: str,
    run_name: Optional[str] = None,
    output_dir: str = "/kaggle/working/ordinal_ngcf_method2",
) -> ExperimentConfig:
    """
    Creates a ready-to-run Ordinal NGCF config.

    weight_fn:
      - "linear_norm"      fixed ordinal baseline
      - "exp_mixture"      learnable smooth monotone function
      - "softplus_spline"  learnable flexible monotone spline
    """
    if weight_fn not in {"linear_norm", "exp_mixture", "softplus_spline"}:
        raise ValueError("weight_fn must be one of: linear_norm, exp_mixture, softplus_spline")

    cfg = ExperimentConfig()

    # =========================
    # paths
    # =========================
    cfg.paths.dataset_dir = "/kaggle/input/datasets/daniilzagatin/movielens20-withfeatures-split"
    cfg.paths.output_dir = output_dir

    # =========================
    # model
    # =========================
    cfg.model.model_type = "ordinal_ngcf"

    # =========================
    # item features
    # =========================
    cfg.features.use_item_features = True
    cfg.features.item_feature_mode = "init_only"  # features initialize item embeddings once; no feature projection during forward
    cfg.features.scaler_kind = "standard"
    cfg.features.fit_scaler_on = "warm"
    cfg.features.feature_dropout = 0.10
    cfg.features.feature_hidden_dim = 256

    # =========================
    # train
    # =========================
    cfg.train.seed = 42
    cfg.train.device = "cuda:0" if torch.cuda.is_available() else "cpu"
    cfg.train.epochs = 15
    cfg.train.lr = 3e-4
    cfg.train.weight_decay = 1e-5
    cfg.train.batch_size = 65536
    cfg.train.num_workers = 2
    cfg.train.patience = 3
    cfg.train.grad_clip_norm = 5.0
    cfg.train.log_every_n_steps = 100
    cfg.train.verbose = True

    # BPR positives. Rating enters the graph through w(r_uv), so sample weighting is disabled here.
    cfg.train.positive_threshold = 4.0
    cfg.train.train_rating_weighting = "none"

    # Full-graph sparse path.
    cfg.train.graph_loss_batches_per_epoch_log = 10
    cfg.train.use_amp = False
    cfg.train.tf32 = True
    cfg.train.use_multi_gpu = False
    cfg.train.compile_model = False

    cfg.train.pin_memory = True
    cfg.train.persistent_workers = False
    cfg.train.prefetch_factor = 2
    cfg.train.empty_cache_each_epoch = False

    # =========================
    # eval
    # =========================
    cfg.eval.ks = (10, 20, 50)
    cfg.eval.metric_gain_types = ("binary_ge_4", "centered_3", "power")
    cfg.eval.binary_metric_names = ("precision", "recall", "hitrate", "mrr", "map", "ndcg")
    cfg.eval.graded_metric_names = ("ndcg",)

    cfg.eval.positive_threshold = 4.0
    cfg.eval.neutral_rating = 3.0
    cfg.eval.power_beta = 0.2
    cfg.eval.power_gamma = 2.0

    cfg.eval.exclude_seen_items = True
    cfg.eval.user_batch_size = 256
    cfg.eval.item_batch_size = 65536
    cfg.eval.compute_loss = False
    cfg.eval.loss_batch_size = 4096

    # Main early-stopping metric.
    cfg.eval.main_metric_gain_type = "centered_3"
    cfg.eval.main_metric_name = "ndcg"
    cfg.eval.main_metric_k = 10

    # Cold users are supported by a fixed shared initialization, without a support encoder.
    cfg.eval.eval_scenarios = ("warm", "cold_user", "cold_item", "both_cold")
    cfg.eval.evaluate_cold_on_val = True
    cfg.eval.evaluate_cold_on_test = True

    # =========================
    # graph / NGCF
    # =========================
    cfg.model.graph.num_layers = 2
    cfg.model.graph.embedding_dim = 128
    cfg.model.graph.hidden_dim = 128
    cfg.model.graph.dropout = 0.10

    cfg.model.graph.rating_weight_fn = weight_fn
    cfg.model.graph.rating_r_min = 0.5
    cfg.model.graph.rating_r_max = 5.0
    cfg.model.graph.rating_weight_eps = 1e-3

    # Optional time decay can be enabled later, but for clean method 2 it is off.
    cfg.model.graph.relation_apply_time_decay = False
    cfg.model.graph.time_weight_fn = "none"
    cfg.model.graph.time_weight_gamma = 0.0

    # exp_mixture hyperparameters.
    cfg.model.graph.rating_exp_terms = 5
    cfg.model.graph.rating_exp_lambda_clip = 8.0

    # softplus_spline hyperparameters.
    cfg.model.graph.rating_softplus_num_knots = 5
    cfg.model.graph.rating_softplus_init_kappa = 8.0
    cfg.model.graph.rating_softplus_learn_kappa = True

    # NGCF-specific pooling.
    cfg.model.graph.ngcf_layer_pooling = "concat"
    cfg.model.graph.ngcf_use_l2_norm = True
    cfg.model.graph.checkpoint_encoder_layers = True

    # New users: no support encoder. All cold users receive the same fixed vector.
    # Better than zero: initialize it as a weighted mean of train positive item embeddings.
    cfg.model.graph.cold_user_init_strategy = "positive_item_mean"

    # =========================
    # comet
    # =========================
    cfg.comet.enabled = True
    cfg.comet.project_name = os.getenv("COMET_PROJECT_NAME", "explicit-movie-ranking")
    cfg.comet.experiment_name = run_name or f"ordinal_ngcf_{weight_fn}"
    cfg.comet.tags = [
        "ordinal-ngcf",
        "method2",
        "rating-as-edge-weight",
        weight_fn,
        "warm",
        "cold_user_fixed",
        "cold_item",
        "both_cold_fixed",
    ]

    return cfg

In [ ]:
# cfg = make_ordinal_ngcf_cfg("linear_norm", run_name="ordinal_ngcf_linear_norm")
# model, data, history_df, best_val_metrics, test_metrics = run_training(cfg)

# cfg = make_ordinal_ngcf_cfg("exp_mixture", run_name="ordinal_ngcf_exp_mixture")
# model, data, history_df, best_val_metrics, test_metrics = run_training(cfg)

# cfg = make_ordinal_ngcf_cfg("softplus_spline", run_name="ordinal_ngcf_softplus_spline")
# model, data, history_df, best_val_metrics, test_metrics = run_training(cfg)

In [ ]:
# ============================================================
# Memory-safe grouped propagation for ordinal_ngcf
# Fixes OOM for exp_mixture / softplus_spline
# ============================================================

import gc
import numpy as np
import torch
import torch.nn.functional as F


def _safe_empty_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _make_sparse_coo_torch(indices_np, values_np, size, device):
    idx = torch.tensor(indices_np, dtype=torch.long, device=device)
    vals = torch.tensor(values_np, dtype=torch.float32, device=device)
    return torch.sparse_coo_tensor(idx, vals, size=size, device=device).coalesce()


def _ordinal_ngcf_set_grouped_graph_edges(self, train_df):
    """
    Memory-safe graph storage.

    Вместо одного sparse adjacency с learnable values строим фиксированные
    adjacency matrices по каждому rating value:

        A_r

    А обучаемый вес w(r) применяется как скаляр перед A_r @ H.
    """
    device = next(self.parameters()).device

    edge_user = torch.tensor(
        train_df["user_idx"].astype(np.int64).to_numpy(),
        dtype=torch.long,
        device=device,
    )
    edge_item = torch.tensor(
        train_df["item_idx"].astype(np.int64).to_numpy(),
        dtype=torch.long,
        device=device,
    )
    edge_rating = torch.tensor(
        train_df["rating"].astype(np.float32).to_numpy(),
        dtype=torch.float32,
        device=device,
    )

    if edge_user.numel() > 0 and int(edge_user.max().item()) >= self.num_warm_users:
        raise ValueError(
            "OrdinalNGCFModel expects train graph to contain only warm users. "
            f"Found max user_idx={int(edge_user.max().item())}, "
            f"num_warm_users={self.num_warm_users}."
        )

    if "timestamp" in train_df.columns:
        edge_timestamp = torch.tensor(
            train_df["timestamp"].astype(np.int64).to_numpy(),
            dtype=torch.long,
            device=device,
        )
        latest_ts = int(train_df["timestamp"].max())
    else:
        edge_timestamp = torch.zeros_like(edge_user)
        latest_ts = 0

    # Сохраняем старые buffers для диагностики / cold-user init.
    self._set_buffer("edge_user", edge_user)
    self._set_buffer("edge_item", edge_item)
    self._set_buffer("edge_rating", edge_rating)
    self._set_buffer("edge_timestamp", edge_timestamp)
    self._set_buffer(
        "latest_timestamp",
        torch.tensor(latest_ts, dtype=torch.long, device=device),
    )

    all_u = train_df["user_idx"].astype(np.int64).to_numpy()
    all_i = train_df["item_idx"].astype(np.int64).to_numpy()
    all_r = train_df["rating"].astype(np.float32).to_numpy()

    # Фиксированная нормализация.
    # Веса w(r) будут учиться отдельно как скаляры.
    base_w = np.ones(len(train_df), dtype=np.float32)

    if getattr(self.graph_cfg, "relation_apply_time_decay", False) and "timestamp" in train_df.columns:
        ts = train_df["timestamp"].astype(np.int64).to_numpy()
        delta = np.maximum(latest_ts - ts, 0).astype(np.float32)
        base_w *= time_weight(
            delta,
            self.graph_cfg.time_weight_fn,
            self.graph_cfg.time_weight_gamma,
        ).astype(np.float32)

    user_deg = np.bincount(
        all_u,
        weights=base_w,
        minlength=self.num_users,
    ).astype(np.float32)

    item_deg = np.bincount(
        all_i,
        weights=base_w,
        minlength=self.num_items,
    ).astype(np.float32)

    rating_values_np = (
        self.edge_weight_fn.rating_values
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    self._set_buffer(
        "group_rating_values",
        torch.tensor(rating_values_np, dtype=torch.float32, device=device),
    )

    group_adjs_ui = []
    group_adjs_iu = []

    for rating in rating_values_np:
        mask = np.isclose(all_r, rating)

        if not np.any(mask):
            empty_idx = np.zeros((2, 0), dtype=np.int64)
            empty_vals = np.zeros((0,), dtype=np.float32)

            group_adjs_ui.append(
                _make_sparse_coo_torch(
                    empty_idx,
                    empty_vals,
                    (self.num_users, self.num_items),
                    device,
                )
            )
            group_adjs_iu.append(
                _make_sparse_coo_torch(
                    empty_idx,
                    empty_vals,
                    (self.num_items, self.num_users),
                    device,
                )
            )
            continue

        u = all_u[mask]
        i = all_i[mask]
        vals = base_w[mask] / np.sqrt(
            np.maximum(user_deg[u], 1e-8) * np.maximum(item_deg[i], 1e-8)
        )
        vals = vals.astype(np.float32)

        ui_idx = np.vstack([u, i])
        iu_idx = np.vstack([i, u])

        group_adjs_ui.append(
            _make_sparse_coo_torch(
                ui_idx,
                vals,
                (self.num_users, self.num_items),
                device,
            )
        )
        group_adjs_iu.append(
            _make_sparse_coo_torch(
                iu_idx,
                vals,
                (self.num_items, self.num_users),
                device,
            )
        )

    # Не register_buffer, потому что это список sparse tensors.
    # Главное: создаётся уже после .to(device).
    self.group_adjs_ui = group_adjs_ui
    self.group_adjs_iu = group_adjs_iu


def _ordinal_ngcf_encode_full_graph_grouped(self):
    """
    Memory-safe grouped ordinal NGCF propagation:

        neighbor_u = sum_r w(r) A_ui_r item_h
        neighbor_i = sum_r w(r) A_iu_r user_h

    Gradients go through only K scalar weights w(r), not through every edge.
    """
    if not hasattr(self, "group_adjs_ui"):
        raise RuntimeError("Grouped graph is not set. Call model.set_graph_edges(train_df).")

    user_h = self.initial_user_vectors()
    item_h = self.item_encoder.all_item_vectors()

    user_states = [user_h]
    item_states = [item_h]

    rating_values = self.group_rating_values.to(
        device=user_h.device,
        dtype=user_h.dtype,
    )

    # Важно: тут есть gradient по параметрам exp_mixture / softplus_spline.
    rating_weights = self.edge_weight_fn(rating_values)

    for layer in self.layers:
        user_neigh = torch.zeros_like(user_h)
        item_neigh = torch.zeros_like(item_h)

        for k, (a_ui, a_iu) in enumerate(zip(self.group_adjs_ui, self.group_adjs_iu)):
            wk = rating_weights[k]

            user_neigh = user_neigh + wk * torch.sparse.mm(a_ui, item_h)
            item_neigh = item_neigh + wk * torch.sparse.mm(a_iu, user_h)

        user_out = layer.W1(user_neigh) + layer.W2(user_h * user_neigh)
        item_out = layer.W1(item_neigh) + layer.W2(item_h * item_neigh)

        user_out = layer.dropout(layer.act(user_out))
        item_out = layer.dropout(layer.act(item_out))

        if layer.use_l2_norm:
            user_out = F.normalize(user_out, p=2, dim=-1)
            item_out = F.normalize(item_out, p=2, dim=-1)

        user_h, item_h = user_out, item_out

        user_states.append(user_h)
        item_states.append(item_h)

    if self.graph_cfg.ngcf_layer_pooling == "concat":
        user_z = torch.cat(user_states, dim=-1)
        item_z = torch.cat(item_states, dim=-1)
    elif self.graph_cfg.ngcf_layer_pooling == "mean":
        user_z = torch.stack(user_states, dim=0).mean(dim=0)
        item_z = torch.stack(item_states, dim=0).mean(dim=0)
    else:
        raise ValueError(f"Unknown ngcf_layer_pooling={self.graph_cfg.ngcf_layer_pooling}")

    return user_z, item_z


# Monkey patch existing class.
OrdinalNGCFModel.set_graph_edges = _ordinal_ngcf_set_grouped_graph_edges
OrdinalNGCFModel.encode_full_graph = _ordinal_ngcf_encode_full_graph_grouped

_safe_empty_cache()

print("Patched OrdinalNGCFModel: grouped rating propagation enabled.")

In [ ]:
# ============================================================
# Memory-safe fixed cold-user initialization for OrdinalNGCF
# ============================================================

import gc
import numpy as np
import torch
import torch.nn.functional as F


@torch.no_grad()
def _initialize_fixed_cold_user_vector_chunked(
    self,
    train_df,
    positive_threshold: float = 4.0,
    chunk_size: int = 262144,
):
    """
    Memory-safe cold-user init.

    Вместо того чтобы брать все positive item vectors сразу,
    считаем weighted mean по чанкам.

    cold_user_vec = sum_edges w(r_ui) * item_init_i / sum_edges w(r_ui)
    """

    device = next(self.parameters()).device

    if not hasattr(self, "fixed_cold_user_vector"):
        self.register_buffer(
            "fixed_cold_user_vector",
            torch.zeros(1, self.graph_cfg.embedding_dim, device=device),
            persistent=True,
        )

    if train_df is None or len(train_df) == 0:
        nn_std = float(self.user_emb.weight.std().detach().cpu())
        cold_vec = torch.randn(
            1,
            self.graph_cfg.embedding_dim,
            device=device,
        ) * max(nn_std, 1e-3)

        self.fixed_cold_user_vector.copy_(cold_vec)
        return

    pos_df = train_df[train_df["rating"] >= positive_threshold]

    if len(pos_df) == 0:
        cold_vec = self.user_emb.weight[: self.num_warm_users].mean(dim=0, keepdim=True)
        self.fixed_cold_user_vector.copy_(cold_vec)
        return

    item_idx_np = pos_df["item_idx"].astype(np.int64).to_numpy()
    rating_np = pos_df["rating"].astype(np.float32).to_numpy()

    # Initial item vectors: именно init-only/item embedding пространство.
    item_init = self.item_encoder.all_item_vectors()

    sum_vec = torch.zeros(
        1,
        self.graph_cfg.embedding_dim,
        device=device,
        dtype=item_init.dtype,
    )
    sum_w = torch.zeros(
        1,
        device=device,
        dtype=item_init.dtype,
    )

    n = len(item_idx_np)

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)

        item_idx = torch.tensor(
            item_idx_np[start:end],
            dtype=torch.long,
            device=device,
        )

        ratings = torch.tensor(
            rating_np[start:end],
            dtype=torch.float32,
            device=device,
        )

        selected = item_init[item_idx]

        weights = self.edge_weight_fn(ratings).detach().clamp_min(1e-8).to(item_init.dtype)
        weights = weights.view(-1, 1)

        sum_vec += (selected * weights).sum(dim=0, keepdim=True)
        sum_w += weights.sum()

        del item_idx, ratings, selected, weights

    cold_vec = sum_vec / sum_w.clamp_min(1e-8)

    # Масштабируем примерно к warm user embeddings.
    warm_user_std = self.user_emb.weight[: self.num_warm_users].std().clamp_min(1e-6)
    cold_std = cold_vec.std().clamp_min(1e-6)
    cold_vec = cold_vec * (warm_user_std / cold_std)

    self.fixed_cold_user_vector.copy_(cold_vec)

    del item_init, sum_vec, sum_w, cold_vec
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


OrdinalNGCFModel.initialize_fixed_cold_user_vector = _initialize_fixed_cold_user_vector_chunked

gc.collect()
torch.cuda.empty_cache()

print("Patched OrdinalNGCFModel.initialize_fixed_cold_user_vector: chunked version enabled.")

In [ ]:
RUN_ALL_ORDINAL_NGCF = True

if RUN_ALL_ORDINAL_NGCF:
    all_results = {}

    for weight_fn in ["softplus_spline"]:
        cfg = make_ordinal_ngcf_cfg(
            weight_fn=weight_fn,
            run_name=f"ordinal_ngcf_{weight_fn}",
        )

        model, data, history_df, best_val_metrics, test_metrics = run_training(cfg)

        all_results[weight_fn] = {
            "history": history_df,
            "best_val_metrics": best_val_metrics,
            "test_metrics": test_metrics,
            "rating_weight_table": get_rating_weight_table_metrics(model),
        }

    all_results

In [ ]:
# get_rating_weight_table_metrics(model)